# OZON E-CUP 2026 · Search LTV
## Отчёт по разведочному анализу

**Задача.** Предсказать суммарный GMV пользователя (Поиск + Каталог) за 30 дней
**2026-02-14 … 2026-03-15**, имея историю **2025-01-01 … 2026-02-13**:
30 631 006 дневных записей по 250 000 пользователей. Метрика — RMSLE, public/private = 20/80.

Это выжимка из полного разбора ([01_eda.ipynb](01_eda.ipynb)): здесь только то, что меняет
решения. Ноутбук самодостаточен — пересчитывает все числа сам, ничего не берёт на веру.

### Словарь: что означают слова, которыми дальше всё описано

Эти четыре понятия используются в каждом разделе, поэтому зафиксируем их сразу.

**Якорь (anchor)** — дата, на которую мы мысленно «встаём». Всё, что произошло **до и включая**
якорь, разрешено использовать как признаки; всё, что после — предмет предсказания.
Якорь — это способ превратить один временной ряд в обычную табличную задачу
«строка = пользователь, признаки → таргет».

**Таргет** для якоря $T$ — сумма GMV пользователя за 30 дней **после** якоря:

$$y_i(T) \;=\; \sum_{t \,=\, T+1}^{T+30} \mathrm{gmv}_{i,t}$$

**Окно признаков** — сколько истории до якоря мы смотрим. Победитель Google Analytics
Customer Revenue брал 168 дней; мы отталкиваемся от той же величины.

Нам важны ровно два якоря:

| Якорь | Дата | Окно таргета | Зачем нужен |
|---|---|---|---|
| **Боевой** | 2026-02-13 | 2026-02-14 … 03-15 | то, что оценивает лидерборд; таргет невидим |
| **Валидационный** | 2026-01-14 | 2026-01-15 … 02-13 | последний якорь, у которого **всё** окно таргета внутри наших данных, — значит на нём можно честно померить качество |

Валидационный якорь выбран не произвольно: он максимально поздний из тех, где таргет
известен полностью. Чем ближе валидация к боевому якорю, тем меньше разница в сезоне,
составе аудитории и уровне спроса.

**Recency** — сколько дней прошло с последнего события пользователя на момент якоря.
**Hurdle-модель** («модель с барьером») — разложение предсказания на две части:
сначала «купит ли вообще», затем «на сколько, если купит».

---
### Резюме: пять выводов, каждый со своим следствием

| Что нашли | Что это меняет |
|---|---|
| **Выборка условна на недавней активности.** Максимальный recency ровно 29 дней; все 250 000 имеют событие в каждом из трёх последних 30-дневных блоков | Обучающие якоря нельзя брать «как есть»: на них попадают пользователи, которых в тесте физически нет, и вероятность покупки смещается вниз |
| **`sample_submit.csv` = точный факт GMV за предыдущее окно** (2026-01-15 … 02-13) | Бесплатный бейзлайн и ориентир по масштабу сабмита: доля ненулевых и средний `log1p` |
| **Под RMSLE hurdle склеивается точно:** $\hat y = e^{p\,m} - 1$ | Привычная склейка $p\cdot(e^m-1)$ хуже на **0.37 RMSLE** — при том что модель одна и та же |
| **Весь сигнал в «купит ли», а не в «на сколько».** $p(x)$ пробегает 8-кратный диапазон, $m(x)$ — менее чем двукратный | Ёмкость модели и время разработки — в классификатор |
| **Валидационное окно — сезонная яма:** −19 % к тренду, самый слабый месяц за всю историю; боевое окно годом ранее было на ~12 % сильнее соседнего январского | Поправка на уровень окна почти бесплатна и даёт +0.005…+0.010 RMSLE — порядка порога значимости, не больше (раздел 8.3) |
| **`средний чек` и `доля Поиска` меняют знак при контроле активности** | Признаки полезные, но только рядом с контролями; как самостоятельный сигнал вводят в заблуждение |

---
## 0. Подготовка данных

Два технических решения, без которых остальное неудобно считать:

* **Даункаст типов.** Исходник читается как `Int64/Float64` и занимает 4090 MB —
  на машине с 18 GB это половина памяти. Приведение флагов к `Int8`, счётчиков к `UInt16`,
  денег к `Float32` даёт 1139 MB, то есть в 3.6 раза меньше, без потери точности
  (максимальный GMV за день — 73 830, `Float32` держит 7 значащих цифр).
* **Целочисленный индекс дня** $d = (\text{дата} - \text{2025-01-01})$ в днях, $d \in [0, 408]$.
  Любое оконное условие превращается в сравнение двух `Int16` вместо арифметики дат —
  агрегации по 30 млн строк считаются секунды.

In [1]:
import gc, warnings
from datetime import date, timedelta
from pathlib import Path

import numpy as np
import polars as pl
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

warnings.filterwarnings("ignore")
pl.Config.set_tbl_rows(20)

ROOT = Path.cwd()
while not (ROOT / "data_start").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
DATA_START, DATA_WORK = ROOT / "data_start", ROOT / "data_work"
DATA_WORK.mkdir(exist_ok=True)

DAY0, DAY_LAST = date(2025, 1, 1), date(2026, 2, 13)
N_DAYS = (DAY_LAST - DAY0).days + 1
HORIZON = 30
ANCHOR_FINAL = N_DAYS - 1             # 408 — боевой якорь
ANCHOR_VAL = ANCHOR_FINAL - HORIZON   # 378 — валидационный якорь
ANCHOR_TRAIN = ANCHOR_VAL - HORIZON   # 348 — обучающий якорь для проверок
RNG = np.random.default_rng(42)

SCHEMA_CAST = {
    "user_id": pl.UInt32, "search": pl.Int8, "cat": pl.Int8,
    "has_search_to_cart": pl.Int8, "has_search_to_ord": pl.Int8,
    "has_cat_to_cart": pl.Int8, "has_cat_to_ord": pl.Int8,
    "search_to_cart": pl.UInt16, "search_to_ord": pl.UInt16,
    "cat_to_cart": pl.UInt16, "cat_to_ord": pl.UInt16,
    "to_cart": pl.UInt16, "to_ord": pl.UInt16, "searches": pl.UInt16,
    "gmv_search": pl.Float32, "gmv_cat": pl.Float32, "gmv": pl.Float32,
}
CACHE = DATA_WORK / "train_compact.parquet"
if CACHE.exists():
    df = pl.read_parquet(CACHE)
else:
    raw = pl.read_parquet(DATA_START / "train.parquet")
    df = (raw.with_columns([pl.col(c).cast(t) for c, t in SCHEMA_CAST.items()])
             .with_columns(d=((pl.col("event_date") - pl.lit(DAY0)).dt.total_days()).cast(pl.Int16),
                           dow=(pl.col("event_date").dt.weekday() - 1).cast(pl.Int8))
             .drop("event_date").sort(["user_id", "d"]))
    del raw; _ = gc.collect()
    df.write_parquet(CACHE, compression="zstd")

USERS = df["user_id"].unique().sort()
N_USERS = USERS.len()
print(f"{df.height:,} строк · {N_USERS:,} пользователей · {N_DAYS} дней · "
      f"{df.estimated_size('mb'):,.0f} MB в памяти")

30,631,006 строк · 250,000 пользователей · 409 дней · 1,139 MB в памяти


In [2]:
# ------------------------------------------------------------------ оформление
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100",
          "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
SEQ_BLUE = ["#cde2fb", "#b7d3f6", "#9ec5f4", "#86b6ef", "#6da7ec", "#5598e7",
            "#3987e5", "#2a78d6", "#256abf", "#1c5cab", "#184f95", "#104281", "#0d366b"]
SEQ = [[i / (len(SEQ_BLUE) - 1), c] for i, c in enumerate(SEQ_BLUE)]
DIVERGING = [[0.0, "#0d366b"], [0.25, "#5598e7"], [0.5, "#f0efec"],
             [0.75, "#e34948"], [1.0, "#8f1f1f"]]
STATUS = {"good": "#0ca30c", "warning": "#fab219", "critical": "#d03b3b"}
SURFACE, INK, INK2, MUTED, GRID, AXIS = "#fcfcfb", "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"

# пропорции под слайд: ближе к 3:2, а не к вытянутой ленте
FIG_W, FIG_H = 960, 580

pio.templates["ecup"] = go.layout.Template(layout=dict(
    colorway=SERIES, paper_bgcolor=SURFACE, plot_bgcolor=SURFACE,
    font=dict(family="system-ui, -apple-system, Segoe UI, sans-serif", size=13, color=INK2),
    title=dict(font=dict(size=17, color=INK), x=0.0, xanchor="left", pad=dict(b=16)),
    margin=dict(l=64, r=32, t=72, b=120),
    xaxis=dict(gridcolor=GRID, zerolinecolor=AXIS, linecolor=AXIS, showline=True,
               tickfont=dict(color=MUTED, size=12), ticks="outside", ticklen=4,
               tickcolor=AXIS, automargin=True, title_font=dict(size=13, color=INK2)),
    yaxis=dict(gridcolor=GRID, zerolinecolor=AXIS, linecolor=AXIS, showline=False,
               tickfont=dict(color=MUTED, size=12), automargin=True,
               title_font=dict(size=13, color=INK2)),
    # легенда под графиком: на y=1.02 она садилась ровно на двухстрочный заголовок
    legend=dict(orientation="h", yanchor="top", y=-0.18, x=0, xanchor="left",
                font=dict(size=12, color=INK2), bgcolor="rgba(0,0,0,0)"),
    hovermode="x unified", colorscale=dict(sequential=SEQ, diverging=DIVERGING)))
pio.templates.default = "ecup"


def show(fig, title=None, sub=None, w=None, h=None):
    """Единый финиш. Размеры подобраны под вставку в слайд, а не под ширину экрана."""
    if title:
        t = title if sub is None else (
            f"{title}<br><span style='font-size:12.5px;color:{MUTED}'>{sub}</span>")
        fig.update_layout(title_text=t, margin=dict(t=92 if sub else 72))
    fig.update_xaxes(automargin=True)
    fig.update_yaxes(automargin=True)
    # без легенды нижнее поле не нужно — не оставляем пустоту под графиком
    if fig.layout.showlegend is False:
        fig.update_layout(margin=dict(b=64))
    fig.update_layout(width=w or FIG_W, height=h or FIG_H, autosize=False)
    fig.show()


def line(fig, x, y, name, slot=0, width=2.5, dash=None, **kw):
    fig.add_trace(go.Scatter(x=x, y=y, name=name, mode="lines",
                             line=dict(color=SERIES[slot % 8], width=width, dash=dash), **kw))


def bars(x, y, slot=0, name="", text=None, orientation="v"):
    return go.Bar(x=x, y=y, name=name, orientation=orientation,
                  marker=dict(color=SERIES[slot % 8], cornerradius=4,
                              line=dict(color=SURFACE, width=1)),
                  text=text, textposition="outside",
                  textfont=dict(color=INK2, size=11), cliponaxis=False)


def hist_trace(v, bins=60, slot=0, name=""):
    v = np.asarray(v, "float64"); v = v[np.isfinite(v)]
    cnt, e = np.histogram(v, bins=bins)
    return go.Bar(x=(e[:-1] + e[1:]) / 2, y=cnt, name=name,
                  marker=dict(color=SERIES[slot % 8], line=dict(color=SURFACE, width=0.5)),
                  hovertemplate="%{x:.3f} → %{y:,}<extra></extra>")


def heat(z, x, y, colorscale=SEQ, zmid=None, hover="%{z:.3f}"):
    return go.Heatmap(z=z, x=x, y=y, colorscale=colorscale, zmid=zmid,
                      hovertemplate="x=%{x}<br>y=%{y}<br>" + hover + "<extra></extra>",
                      colorbar=dict(thickness=12, outlinewidth=0, len=0.85,
                                    tickfont=dict(color=MUTED, size=11)))


def sub_titles(fig, size=12.5):
    for a in fig.layout.annotations:
        a.font = dict(size=size, color=INK2)
    return fig


def rmsle(y_true, y_pred):
    p = np.clip(np.asarray(y_pred, "float64"), 0, None)
    return float(np.sqrt(np.mean((np.log1p(p) - np.log1p(np.asarray(y_true, "float64")))**2)))


def make_target(anchor_d, horizon=HORIZON):
    """Суммарный GMV каждого пользователя за (anchor_d, anchor_d + horizon]."""
    t = (df.filter(pl.col("d").is_between(anchor_d + 1, anchor_d + horizon))
           .group_by("user_id").agg(y=pl.col("gmv").sum().cast(pl.Float64)))
    # left join + fill 0: у неактивных в окне пользователей таргет равен нулю, а не пропуску
    return (pl.DataFrame({"user_id": USERS}).join(t, on="user_id", how="left")
              .with_columns(pl.col("y").fill_null(0.0)).sort("user_id"))


def past_sum(anchor_d, days, col="gmv"):
    """Сумма колонки за последние `days` дней до якоря включительно."""
    s = (df.filter(pl.col("d").is_between(anchor_d - days + 1, anchor_d))
           .group_by("user_id").agg(v=pl.col(col).sum().cast(pl.Float64)))
    return (pl.DataFrame({"user_id": USERS}).join(s, on="user_id", how="left")
              .with_columns(pl.col("v").fill_null(0.0)).sort("user_id")["v"].to_numpy())


tgt = make_target(ANCHOR_VAL)
y = tgt["y"].to_numpy()
ly = np.log1p(y)
print(f"валидационный якорь {DAY0+timedelta(days=ANCHOR_VAL)} · "
      f"таргет-окно {DAY0+timedelta(days=ANCHOR_VAL+1)} … {DAY0+timedelta(days=ANCHOR_VAL+HORIZON)}")
print(f"таргет посчитан для всех {len(y):,} пользователей "
      f"(у {100*(y==0).mean():.1f}% он равен нулю)")

валидационный якорь 2026-01-14 · таргет-окно 2026-01-15 … 2026-02-13
таргет посчитан для всех 250,000 пользователей (у 45.9% он равен нулю)


---
# 1. Метрика: что на самом деле оптимизируется

Разбор метрики стоит первым не для формальности. Три четверти неверных решений в этой
задаче — не «слабая модель», а модель, обученная оптимизировать не то. Поэтому сначала
разберёмся, что именно RMSLE считает хорошим ответом.

$$
\mathrm{RMSLE} \;=\; \sqrt{\frac{1}{n}\sum_{i=1}^{n}\Big(\log(1+\hat y_i) - \log(1+y_i)\Big)^{2}}
$$

Замена переменной $z_i = \log(1+y_i)$, $\hat z_i = \log(1+\hat y_i)$ превращает её в обычный RMSE:

$$
\mathrm{RMSLE}(\hat y, y) \;=\; \mathrm{RMSE}(\hat z, z)
$$

Дальше всё выводится в две строки — и почти каждый вывод противоречит интуиции.

### 1.1 Ошибка измеряется в разах, а не в рублях

$$
\log(1+\hat y) - \log(1+y) \;=\; \log\frac{1+\hat y}{1+y}
$$

Штраф зависит **только от отношения** предсказания к факту. Промах «200 вместо 100»
и промах «20 000 вместо 10 000» стоят ровно одинаково.

*Что это даёт.* Крупные покупатели не доминируют в функции потерь, и отдельно бороться
за них бессмысленно. Зато ошибка на пользователе с нулевым или копеечным GMV стоит
столько же, сколько на топовом — а таких пользователей у нас почти половина.
Вся ценность модели сосредоточена там, где бизнес-интуиция подсказывает обратное.

### 1.2 Оптимальная константа — не среднее

Если бы мы предсказывали всем одно и то же число $c$, какое было бы лучшим?

$$
\frac{\partial}{\partial c}\sum_i\big(\log(1+c)-z_i\big)^2 = 0
\;\Longrightarrow\;
\log(1+c^{*}) = \bar z
\;\Longrightarrow\;
\boxed{\,c^{*} = e^{\bar z} - 1 = \Big(\textstyle\prod_i (1+y_i)\Big)^{1/n} - 1\,}
$$

Это **геометрическое** среднее от $(1+y)$, а не арифметическое от $y$. Разница
на наших данных — на порядок, и она стоит почти целой единицы RMSLE.

### 1.3 Оптимальный прогноз — экспонента среднего логарифма

$$
\hat y^{*}(x) \;=\; \exp\!\big(\mathbb{E}[\log(1+y)\mid x]\big) - 1
\;\underset{\text{неравенство Йенсена}}{\le}\; \mathbb{E}[y \mid x]
$$

*Что это даёт.* Модель, честно предсказывающая условное **среднее** $\mathbb{E}[y\mid x]$
(а это то, что делает любая регрессия с MSE в исходной шкале), под RMSLE систематически
завышает — тем сильнее, чем длиннее хвост распределения. Единственный способ не наступить
на эти грабли: обучать регрессию **прямо в `log1p`-пространстве**.

In [3]:
best_const = np.expm1(ly.mean())
print(f"доля нулей в таргете          : {(y==0).mean():.4f}")
print(f"арифметическое среднее y      : {y.mean():.3f}")
print(f"оптимальная константа c*      : {best_const:.3f}")
print(f"RMSLE(c*)                     : {rmsle(y, np.full_like(y, best_const)):.4f}")
print(f"RMSLE(среднее y)              : {rmsle(y, np.full_like(y, y.mean())):.4f}   "
      f"← дороже на {rmsle(y, np.full_like(y, y.mean())) - rmsle(y, np.full_like(y, best_const)):.4f}")

# диапазон подрезан вокруг содержательной области: нуль на логарифмической оси
# не рисуется и растягивал бы её в пустоту
LO, HI = 0.5, 400.0
cs = np.logspace(np.log10(LO), np.log10(HI), 260)
sc = [rmsle(y, np.full_like(y, c)) for c in cs]
fig = go.Figure()
line(fig, cs, sc, "RMSLE", slot=0)
fig.add_vline(x=best_const, line=dict(color=STATUS["good"], width=2),
              annotation_text=f"c* = {best_const:.1f}", annotation_position="top left",
              annotation_font=dict(color=STATUS["good"], size=12))
fig.add_vline(x=y.mean(), line=dict(color=STATUS["critical"], width=2, dash="dash"),
              annotation_text=f"среднее y = {y.mean():.0f}", annotation_position="top right",
              annotation_font=dict(color=STATUS["critical"], size=12))
fig.update_xaxes(title="константа прогноза", type="log", range=[np.log10(LO), np.log10(HI)])
fig.update_yaxes(title="RMSLE", range=[min(sc) - 0.05, max(sc) + 0.08])
fig.update_layout(showlegend=False)
show(fig, "Цена неправильного центра",
     sub="ось X логарифмическая",
     w=900, h=560)

доля нулей в таргете          : 0.4593
арифметическое среднее y      : 84.034
оптимальная константа c*      : 8.413
RMSLE(c*)                     : 2.2883
RMSLE(среднее y)              : 3.1750   ← дороже на 0.8867


### 1.4 Сколько шума в лидерборде

Public-часть — 20 % от 250 000, то есть **50 000 пользователей**. Прежде чем радоваться
улучшению на четвёртом знаке, полезно знать, какой разброс даёт одна лишь случайность
разбиения на public/private.

**Как считаем.** Берём один фиксированный прогноз (ошибка каждого пользователя уже
известна и не меняется), 2000 раз случайно набираем подвыборку размера public-части
и смотрим разброс RMSLE. Модель при этом одна и та же — вся вариация здесь
исключительно от того, кто попал в выборку.

In [4]:
pred_ref = np.expm1(0.70 * np.log1p(past_sum(ANCHOR_VAL, 90) / 3))
err2 = (np.log1p(pred_ref) - ly) ** 2                    # поточечная ошибка, фиксирована
boot = np.array([np.sqrt(err2[RNG.integers(0, len(err2), 50_000)].mean()) for _ in range(2000)])
print(f"RMSLE на всех 250K       : {np.sqrt(err2.mean()):.4f}")
print(f"public-подвыборка 50K    : {boot.mean():.4f} ± {boot.std():.4f} (1σ)")
print(f"95% интервал             : [{np.percentile(boot,2.5):.4f}, {np.percentile(boot,97.5):.4f}]")
print(f"\n→ улучшение меньше ~{2*boot.std():.4f} на public неотличимо от шума разбиения")

fig = go.Figure(hist_trace(boot, bins=60, slot=0))
fig.add_vline(x=np.sqrt(err2.mean()), line=dict(color=INK, width=2),
              annotation_text="все 250K", annotation_font=dict(color=INK2, size=11))
fig.update_xaxes(title="RMSLE на случайной подвыборке 50 000")
fig.update_yaxes(title="сколько подвыборок")
fig.update_layout(hovermode="closest", bargap=0.02, showlegend=False)
show(fig, "Шум public-лидерборда при неизменной модели",
     sub="2000 подвыборок по 50 000, модель неизменна",
     w=900, h=540)

RMSLE на всех 250K       : 1.8361
public-подвыборка 50K    : 1.8362 ± 0.0055 (1σ)
95% интервал             : [1.8257, 1.8471]

→ улучшение меньше ~0.0111 на public неотличимо от шума разбиения


*Что это даёт.* Двойное стандартное отклонение — это порог значимости. Всё, что меньше,
не является улучшением, сколько бы оно ни радовало на public. Ориентироваться нужно
на собственную валидацию по времени, а public использовать только как проверку,
что ничего не сломалось на отправке.

---
# 2. Данные: разреженность и концентрация

## 2.1 Насколько данные пусты

Если представить данные как таблицу «250 000 пользователей × 409 дней», получится
102.25 млн ячеек. Записей в файле — 30.6 млн. То есть **строка есть примерно у трети
ячеек**, а остальные две трети — это дни, когда пользователь просто не заходил.

Организаторы предупреждают, что дозаполнение нулями раздует данные до 100+ млн строк.
Ниже видно, зачем вообще об этом думать: полезного сигнала ещё меньше, чем кажется.

In [5]:
grid = N_USERS * N_DAYS
n_rows, n_gmv = df.height, int((df["gmv"] > 0).sum())
stages = [("Полная сетка пользователь × день", grid),
          ("Есть строка — пользователь заходил", n_rows),
          ("В этот день что-то положил в корзину", int((df["to_cart"] > 0).sum())),
          ("В этот день что-то купил (gmv > 0)", n_gmv)]
fig = go.Figure()
for i, (nm, v) in enumerate(stages):
    fig.add_trace(go.Bar(y=[nm], x=[v], orientation="h", showlegend=False,
                         marker=dict(color=SEQ_BLUE[3 + 3 * i], cornerradius=4,
                                     line=dict(color=SURFACE, width=1)),
                         text=[f"  {v/1e6:.1f} млн · {100*v/grid:.1f}% сетки"],
                         textposition="outside", textfont=dict(color=INK2, size=12.5),
                         cliponaxis=False))
fig.update_xaxes(title="ячеек таблицы «пользователь × день»", range=[0, grid * 1.35])
fig.update_yaxes(autorange="reversed")
fig.update_layout(hovermode="closest")
show(fig, "Воронка заполненности данных",
     w=980, h=470)

*Что это даёт.* Дни с покупкой — **4.6 % полной сетки**. Именно из них состоит весь сигнал
о деньгах; всё остальное — контекст. Отсюда два практических следствия: сетку не дозаполняем
(агрегации считаются по прореженной панели), и любую модель проектируем в расчёте на то,
что положительных примеров мало.

Как эта разреженность выглядит вживую — на картинке ниже. Каждая строка — один
пользователь, каждый столбец — один день из 409.

**Почему именно 320 пользователей и как они выбраны.** Отрисовать все 250 000 строк
нельзя — это 102 млн ячеек, браузер такое не покажет. Поэтому берём 320 строк
**равномерно по всему спектру активности**: сортируем пользователей по числу активных
дней и берём каждого 780-го. Так наверху оказываются самые активные, внизу — самые
редкие, и видна вся палитра поведения, а не только типичный случай.

In [6]:
CACHE_U = DATA_WORK / "user_profile.parquet"
if CACHE_U.exists():
    up = pl.read_parquet(CACHE_U)
else:
    up = (df.group_by("user_id").agg(
              n_days=pl.len(), first_d=pl.col("d").min(), last_d=pl.col("d").max(),
              n_ord_days=(pl.col("to_ord") > 0).sum(), n_cart_days=(pl.col("to_cart") > 0).sum(),
              gmv_total=pl.col("gmv").sum(), ord_total=pl.col("to_ord").sum())
            .with_columns(recency=(N_DAYS - 1 - pl.col("last_d")).cast(pl.Int32))
            .sort("user_id"))
    up.write_parquet(CACHE_U, compression="zstd")

n_show = 320
pick = up.sort("n_days", descending=True)                      # от самых активных к самым редким
pick_ids = pick["user_id"].to_numpy()[np.linspace(0, N_USERS - 1, n_show).astype(int)]
src = df.filter(pl.col("user_id").is_in(pick_ids)).select(["user_id", "d", "gmv"])
pos = {u: i for i, u in enumerate(pick_ids)}
Z = np.zeros((n_show, N_DAYS), dtype="float32")                # 0 = строки нет
ri = np.array([pos[u] for u in src["user_id"].to_numpy()])
dd, gg = src["d"].to_numpy(), src["gmv"].to_numpy()
Z[ri, dd] = 1.0                                                # 1 = визит
Z[ri[gg > 0], dd[gg > 0]] = 2.0                                # 2 = визит с покупкой
DATES = [DAY0 + timedelta(days=i) for i in range(N_DAYS)]
fig = go.Figure(go.Heatmap(z=Z, x=DATES, zmin=0, zmax=2, showscale=False,
                           colorscale=[[0.0, SURFACE], [0.49, SURFACE],
                                       [0.5, "#9ec5f4"], [0.74, "#9ec5f4"],
                                       [0.75, "#0d366b"], [1.0, "#0d366b"]],
                           hovertemplate="%{x|%Y-%m-%d}<extra></extra>"))
# autorange="reversed": строка 0 — самый активный пользователь, и она должна быть сверху
fig.update_yaxes(title="320 пользователей, сверху вниз по убыванию активности",
                 showticklabels=False, autorange="reversed")
show(fig, "Как выглядит разреженность вблизи",
     sub="белое — визита нет, голубое — визит, тёмное — визит с покупкой",
     w=1060, h=680)
del Z, src; _ = gc.collect()

Верхняя часть — плотные горизонтальные полосы, это ежедневные пользователи. Нижняя —
редкие точки. Между ними видны обрывы: полоса резко кончается, и дальше белое.
Ровный правый край картинки не случаен — к нему вернёмся в разделе 3.

## 2.2 Концентрация: сколько людей делают весь оборот

Теперь второй вопрос про структуру данных: деньги распределены равномерно
или сосредоточены у немногих?

**Кривая Лоренца** строится так: сортируем пользователей по возрастанию суммарного GMV
и рисуем, какую долю оборота дают беднейшие $u$ процентов. При абсолютном равенстве
получилась бы диагональ. **Коэффициент Джини** — площадь между диагональю и фактической
кривой, нормированная так, что 0 — полное равенство, 1 — весь оборот у одного человека:

$$
G \;=\; 1 - 2\int_0^1 L(u)\,\mathrm{d}u,
\qquad
L(u) = \frac{\sum_{i \le u\,n} y_{(i)}}{\sum_i y_i}
$$

где $y_{(i)}$ — суммарный GMV пользователей, отсортированный по возрастанию.
Интеграл берётся численно, методом трапеций, прямо по кривой — это и есть тот
коэффициент, который печатается в заголовке графика.

**Зачем нам этот график.** Он отвечает на вопрос, который решает выбор функции потерь:
если бы оборот делали немногие, оптимизация в исходной шкале рублей означала бы
подгонку под сотню-другую людей, а на всех остальных модель могла бы выдавать что угодно.

In [7]:
v = np.sort(up["gmv_total"].to_numpy().astype("float64"))      # по возрастанию
cum = np.cumsum(v); cum = cum / cum[-1]                        # L(u): доля оборота
frac = np.arange(1, len(v) + 1) / len(v)                       # u: доля пользователей
gini = 1 - 2 * np.trapezoid(cum, frac)                         # G = 1 − 2∫L(u)du
step = max(1, len(v) // 3000)
fig = go.Figure()
line(fig, frac[::step] * 100, cum[::step] * 100, "факт", slot=0)
fig.add_trace(go.Scatter(x=[0, 100], y=[0, 100], mode="lines", name="равномерное",
                         line=dict(color=AXIS, width=1.5, dash="dash")))
fig.update_xaxes(title="% пользователей по возрастанию GMV")
fig.update_yaxes(title="% суммарного GMV")
show(fig, f"Кривая Лоренца · коэффициент Джини = {gini:.3f}", w=780, h=620)
for p_ in [1, 10, 20, 50]:
    print(f"топ-{p_:<2}% пользователей дают {100*(1-cum[int(len(v)*(1-p_/100))]):5.1f}% GMV")
print(f"\nпользователей вообще без покупок за 409 дней: "
      f"{(up['gmv_total']==0).sum():,} ({100*(up['gmv_total']==0).mean():.1f}%)")

топ-1 % пользователей дают  14.9% GMV
топ-10% пользователей дают  53.0% GMV
топ-20% пользователей дают  71.9% GMV
топ-50% пользователей дают  94.8% GMV

пользователей вообще без покупок за 409 дней: 30,832 (12.3%)


*Что это даёт.* Джини 0.70 — концентрация высокая, но не экстремальная: верхние 10 %
делают чуть больше половины оборота, верхняя половина — 95 %. Ключевой момент в том,
что **метрика этой концентрации не видит**: в логарифмической шкале (раздел 1.1) вклад
топового и рядового пользователя в ошибку одинаков. То есть картинка показывает, где
деньги бизнеса, и одновременно объясняет, почему модель нельзя строить вокруг них.
Оптимизировать надо середину распределения, а не хвост.

---
# 3. Находка №1: выборка отобрана по недавней активности

## 3.1 Что натолкнуло

При построении признака **recency** (сколько дней прошло с последнего события
пользователя на 2026-02-13) обнаружилось, что у него есть жёсткий потолок.
Само по себе это странно: в любой реальной базе есть люди, не заходившие полгода.
Значит выборку нам выдали не случайную, а по какому-то правилу — и это правило надо
восстановить, потому что оно определяет, кто вообще попадает в тест.

## 3.2 Как проверяли

Гипотеза: пользователей отбирали по активности в последних 30-дневных блоках.
Если это так, доля активных в блоках, **выровненных по концу истории**, должна быть
ровно 100 %, а в любых **сдвинутых** окнах той же длины — заметно меньше.
Сдвиг на 5 дней — это контрольная проверка: он ломает выравнивание, не меняя ничего
остального.

In [8]:
r = up["recency"].to_numpy()
print(f"максимальный recency на 2026-02-13 : {r.max()} дней")
print(f"пользователей с recency ≥ 30       : {(r >= 30).sum()}\n")

aligned, shifted = [], []
for A in range(N_DAYS - 1, 40, -30):          # блоки, выровненные от конца истории
    aligned.append((A, df.filter(pl.col("d").is_between(A - 29, A))["user_id"].n_unique() / N_USERS))
for A in range(N_DAYS - 1, N_DAYS - 1 - 45, -5):   # те же окна, но со сдвигом
    shifted.append((A, df.filter(pl.col("d").is_between(A - 29, A))["user_id"].n_unique() / N_USERS))
al, sh = np.array(aligned), np.array(shifted)

fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.11,
                    subplot_titles=["Блоки, выровненные от конца истории",
                                    "Те же окна со сдвигом на 5 дней"])
fig.add_trace(bars([(DAY0 + timedelta(days=int(a))).strftime("%m-%d") for a in al[:, 0]],
                   al[:, 1] * 100, slot=0), row=1, col=1)
fig.add_trace(bars([f"d={int(a)}" for a in sh[:, 0]], sh[:, 1] * 100, slot=1), row=1, col=2)
for c in (1, 2):
    fig.add_hline(y=100, line=dict(color=STATUS["good"], width=1.5, dash="dash"), row=1, col=c)
    fig.update_yaxes(range=[60, 106], title="% активных" if c == 1 else None, row=1, col=c)
    fig.update_xaxes(title="конец окна", row=1, col=c)
sub_titles(fig)
fig.update_layout(showlegend=False, hovermode="closest")
show(fig, "Доля пользователей, активных в 30-дневном окне",
     sub="пунктир — уровень 100%",
     w=1060, h=580)

максимальный recency на 2026-02-13 : 29 дней
пользователей с recency ≥ 30       : 0



## 3.3 Что нашли

Три последних выровненных блока дают **ровно 100.00 %**, любой сдвиг — 98–99 %.
Правило восстанавливается однозначно:

> в выборку вошли пользователи, у которых есть хотя бы одно событие **в каждом** из трёх
> последних 30-дневных блоков: 2025-11-16 … 12-15, 12-16 … 2026-01-14, 01-15 … 02-13.

Проверим формально: запрограммируем правило и применим его к боевому якорю.
Если восстановлено верно, должны остаться ровно все 250 000.

In [9]:
def selected_users(anchor_d, blocks=3, block=30):
    """Пользователи, проходящие правило отбора организаторов на момент anchor_d.

    Требуется хотя бы одно событие в каждом из `blocks` подряд идущих окон
    длины `block`, отсчитанных назад от якоря.
    """
    lo = anchor_d - blocks * block + 1
    s = (df.filter(pl.col("d").is_between(lo, anchor_d))
           .with_columns(b=((pl.col("d") - lo) // block).cast(pl.Int8))
           .group_by("user_id").agg(nb=pl.col("b").n_unique()))
    return s.filter(pl.col("nb") == blocks)["user_id"]


chk = selected_users(ANCHOR_FINAL)
print(f"правилу удовлетворяют {chk.len():,} из {N_USERS:,} → "
      f"{'правило восстановлено верно' if chk.len() == N_USERS else 'НЕ СХОДИТСЯ'}")

sel = np.array([(A, selected_users(A).len() / N_USERS) for A in range(N_DAYS - 1, 90, -15)])
fig = go.Figure()
line(fig, [DAY0 + timedelta(days=int(a)) for a in sel[:, 0]], sel[:, 1] * 100,
     "прошли отбор", slot=0)
fig.add_hline(y=100, line=dict(color=STATUS["good"], width=1.5, dash="dash"))
fig.update_yaxes(title="% от 250 000", range=[60, 106])
fig.update_xaxes(title="якорь")
fig.update_layout(showlegend=False)
show(fig, "Сколько пользователей прошли бы отбор, если встать на прошлый якорь",
     w=920, h=560)

правилу удовлетворяют 250,000 из 250,000 → правило восстановлено верно


## 3.4 Почему это важно и что даёт

На **боевом** якоре фильтр проходят все 250 000 — по построению выборки.
На якоре полугодовой давности — около 72 %.

Значит, если обучаться на всех подряд пользователях исторического якоря, в обучающей
выборке окажется популяция, которой в тесте **физически нет**: люди, давно не заходившие,
у которых таргет почти гарантированно нулевой. Классификатор «купит ли» увидит завышенную
долю нулей, выучит заниженную вероятность покупки и на боевом окне будет систематически
осторожничать. А поскольку RMSLE — метрика калибровки (раздел 1), это прямой проигрыш.

Посмотрим на масштаб: сравним характеристики таргета на каждом якоре для всех
пользователей и для тех, кто прошёл бы отбор.

In [10]:
anchors = list(range(ANCHOR_VAL, 160, -30))[::-1]   # шаг 30 дней назад от валидационного

def drift_row(a, ids=None):
    t = make_target(a)
    if ids is not None:
        t = t.join(pl.DataFrame({"user_id": ids}), on="user_id", how="semi")
    ya = t["y"].to_numpy(); lya = np.log1p(ya)
    return dict(anchor=DAY0 + timedelta(days=a), n=len(ya),
                zero_rate=float((ya == 0).mean()), mean_log1p=float(lya.mean()))

drift = pl.DataFrame([drift_row(a) for a in anchors]).sort("anchor")
drift_sel = pl.DataFrame([drift_row(a, selected_users(a)) for a in anchors]).sort("anchor")

fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.11,
                    subplot_titles=["Доля пользователей с нулевым таргетом",
                                    "Средний log1p(таргета)"])
for cc, col in enumerate(["zero_rate", "mean_log1p"], start=1):
    for dd, nm, slot, dash in [(drift, "все подряд", 7, "dot"),
                               (drift_sel, "с фильтром отбора", 0, None)]:
        fig.add_trace(go.Scatter(x=dd["anchor"].to_list(), y=dd[col].to_numpy(),
                                 mode="lines+markers", name=nm, legendgroup=nm,
                                 showlegend=(cc == 1),
                                 line=dict(color=SERIES[slot], width=2.5, dash=dash),
                                 marker=dict(size=8, line=dict(color=SURFACE, width=2))),
                      row=1, col=cc)
    fig.update_xaxes(title="якорь", row=1, col=cc)
sub_titles(fig)
for a in fig.layout.annotations:
    a.x, a.xanchor = 0, "left"
fig.update_layout(hovermode="x")
show(fig, "Дрейф таргета по якорям: цена игнорирования правила отбора",
     w=1060, h=580)
print(drift_sel)

shape: (8, 4)
┌────────────┬────────┬───────────┬────────────┐
│ anchor     ┆ n      ┆ zero_rate ┆ mean_log1p │
│ ---        ┆ ---    ┆ ---       ┆ ---        │
│ date       ┆ i64    ┆ f64       ┆ f64        │
╞════════════╪════════╪═══════════╪════════════╡
│ 2025-06-18 ┆ 180063 ┆ 0.428084  ┆ 2.465669   │
│ 2025-07-18 ┆ 182541 ┆ 0.410374  ┆ 2.55598    │
│ 2025-08-17 ┆ 185814 ┆ 0.393189  ┆ 2.641523   │
│ 2025-09-16 ┆ 190690 ┆ 0.38937   ┆ 2.643058   │
│ 2025-10-16 ┆ 197379 ┆ 0.387138  ┆ 2.631519   │
│ 2025-11-15 ┆ 205349 ┆ 0.374923  ┆ 2.713424   │
│ 2025-12-15 ┆ 218400 ┆ 0.395421  ┆ 2.614457   │
│ 2026-01-14 ┆ 233152 ┆ 0.438148  ┆ 2.337637   │
└────────────┴────────┴───────────┴────────────┘


Разрыв между пунктиром и сплошной линией и есть систематическое смещение.
На якоре июня 2025 доля нулей отличается на 12 процентных пунктов — это не тонкость,
это другая задача.

**Тонкость, которую легко упустить.** Валидационный якорь 2026-01-14 — тоже не «все
250 000»: фильтр там проходят около 93 %. Оставшиеся 7 % — это люди, которые
на 14 января выглядели бы «выпавшими», и в тестовой популяции таких нет по построению.
Поэтому честная валидационная популяция — результат `selected_users(378)`,
а не вся база. Сдвиг метрики от этого небольшой, но систематический и односторонний.

**Что делать.** На каждом обучающем и валидационном якоре $A$ применять
`selected_users(A)`. Одна строка кода, эффект — на калибровке всей модели.

---
# 4. Находка №2: что лежит в `sample_submit.csv`

## 4.1 Что натолкнуло

Файл-пример обычно заполняют либо нулями, либо случайными числами. Здесь же в нём
осмысленные значения с семью знаками после запятой, и доля ненулевых предсказаний
подозрительно похожа на долю покупающих в наших данных. Проверим его буквально:
сравним с таргетом валидационного якоря — то есть с фактическим GMV
за 2026-01-15 … 02-13, окно ровно перед боевым.

In [11]:
sample = pl.read_csv(DATA_START / "sample_submit.csv").sort("user_id")
sp = sample["predict"].to_numpy()
print(f"состав user_id совпадает с train    : {sample['user_id'].to_list() == USERS.to_list()}")
print(f"max |predict − y(якорь 2026-01-14)| : {np.abs(sp - y).max():.6f}")
print(f"доля точных совпадений              : {np.mean(np.isclose(sp, y, rtol=1e-6, atol=1e-3)):.4f}")
print(f"RMSLE(sample_submit, y)             : {rmsle(y, sp):.8f}")

состав user_id совпадает с train    : True
max |predict − y(якорь 2026-01-14)| : 0.001593
доля точных совпадений              : 1.0000
RMSLE(sample_submit, y)             : 0.00000002


## 4.2 Что это даёт

Совпадение стопроцентное — расхождение только в последнем знаке округления.
`sample_submit.csv` — это **точный факт GMV за предыдущее 30-дневное окно**.

Отсюда два бесплатных вывода:

1. **Готовый бейзлайн.** Отправка файла как есть эквивалентна стратегии «предскажем
   то же, что было в прошлом месяце». Его качество на валидации мы измерим в разделе 6
   и будем считать нижней планкой: решение, которое его не бьёт, бессмысленно.
2. **Ориентир по масштабу сабмита.** Организаторы фактически показали, какого порядка
   распределение они ожидают: доля ненулевых предсказаний и средний `log1p`.
   Если ваш прогноз сильно отличается по этим двум числам — почти наверняка сбита
   калибровка, а не найден новый сигнал.

---
# 5. Находка №3: под RMSLE hurdle склеивается точно

## 5.1 Форма таргета: почему вообще две модели

Посмотрим на распределение таргета. Слева — все пользователи, справа —
**только те, у кого таргет положительный**.

**Зачем нужна вторая панель.** На левой панели 46 % наблюдений сидят в одной точке
(ноль), и этот пик визуально подавляет всё остальное — по нему невозможно понять форму
распределения тех, кто всё-таки купил. Разделив выборку, мы проверяем конкретную
гипотезу: положительная часть логнормальна, то есть в `log1p`-шкале близка к нормальной.
Если это так, то регрессию имеет смысл учить **только на покупавших** и именно
в логарифмах — там она решает простую, почти симметричную задачу вместо кошмара
со смесью распределений.

In [12]:
pos = ly[y > 0]
fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.11,
                    subplot_titles=["Все пользователи: 46% сидят в нуле",
                                    "Только покупавшие: почти логнормально"])
fig.add_trace(hist_trace(ly, bins=80, slot=0), row=1, col=1)
fig.add_trace(hist_trace(pos, bins=80, slot=2), row=1, col=2)
xs = np.linspace(pos.min(), pos.max(), 200)
pdf = np.exp(-0.5 * ((xs - pos.mean()) / pos.std())**2) / (pos.std() * np.sqrt(2 * np.pi))
fig.add_trace(go.Scatter(x=xs, y=pdf * len(pos) * (pos.max() - pos.min()) / 80,
                         mode="lines", name="нормальное приближение",
                         line=dict(color=INK, width=2, dash="dash")), row=1, col=2)
fig.update_yaxes(title="пользователей", row=1, col=1)
for c in (1, 2):
    fig.update_xaxes(title="log1p(GMV за 30 дней)", row=1, col=c)
sub_titles(fig)
fig.update_layout(hovermode="closest", bargap=0.02)
show(fig, "Распределение таргета: смесь точечной массы и логнормали",
     sub=f"нулей {100*(y==0).mean():.1f}%",
     w=1060, h=580)

## 5.2 Точное правило склейки

Итак, две модели: $p(x) = P(y>0\mid x)$ — «купит ли», и
$m(x) = \mathbb{E}[\log(1+y)\mid x,\, y>0]$ — «на сколько в логарифмах, если купит».
Вопрос: как из них собрать один прогноз?

Ключевое обстоятельство — **$\log(1+0) = 0$**. Нулевая компонента вносит в условное
среднее логарифма ровно ноль, поэтому разложение оказывается не приближённым, а точным:

$$
\mathbb{E}[\log(1+y)\mid x]
= p(x)\cdot \underbrace{\mathbb{E}[\log(1+y)\mid x, y>0]}_{m(x)}
\;+\; \big(1-p(x)\big)\cdot \underbrace{0}_{\log(1+0)}
= p(x)\,m(x)
$$

Подставляем в формулу оптимального прогноза из раздела 1.3:

$$
\boxed{\;\hat y^{*}(x) \;=\; \exp\big(p(x)\,m(x)\big) - 1\;}
$$

**Чем это не является.** Интуитивная склейка «вероятность умножить на условную сумму»

$$
\hat y_{\text{наивно}}(x) \;=\; p(x)\cdot\big(e^{m(x)}-1\big)
$$

правильна для MSE в исходной шкале рублей, но **неверна** для RMSLE: она перемножает
величины из разных шкал — вероятность и уже разлогарифмированную сумму.
Разница не косметическая, сейчас измерим.

## 5.3 Численная проверка

**Как устроен эксперимент.** Нужна модель, достаточно простая, чтобы не отвлекать
от сути, но достаточно осмысленная, чтобы разница была не шумом. Берём биннинг:
разбиваем пользователей по трём признакам (recency, число дней с покупкой за 90 дней,
суммарный GMV за 90 дней) на ячейки и в каждой ячейке оцениваем $p$ и $m$ прямым
усреднением.

**Почему обучаем на одном якоре, а проверяем на другом.** Оценки $p$ и $m$ считаются
на якоре 2025-12-15 (окно таргета 2025-12-16 … 2026-01-14), а применяются на якоре
2026-01-14 (окно 2026-01-15 … 02-13). Окна таргета **не пересекаются**, так что
подглядывания в ответ нет. Если бы мы оценили и применили на одном якоре, любая
ячейка идеально «предсказала» бы саму себя, и сравнение формул потеряло бы смысл.

**Зачем сглаживание.** В редких ячейках может оказаться пять человек, и доля покупок
там — чистый шум. Поэтому оценка притягивается к глобальному среднему с весом $K = 30$
(эквивалент тридцати «виртуальных» наблюдений): $\hat p = (n_+ + K\,p_0)/(n + K)$.

In [13]:
def compact_features(anchor_d):
    """Минимальный набор признаков на момент anchor_d: только прошлое, без утечки."""
    h = df.filter(pl.col("d").is_between(anchor_d - 179, anchor_d))
    f = h.group_by("user_id").agg(
        gmv_90=pl.col("gmv").filter(pl.col("d") > anchor_d - 90).sum(),
        gmv_180=pl.col("gmv").sum(),
        ord_days_90=((pl.col("d") > anchor_d - 90) & (pl.col("to_ord") > 0)).sum(),
        act_days_30=(pl.col("d") > anchor_d - 30).sum(),
        r_act=(anchor_d - pl.col("d").max()),
        r_gmv=(anchor_d - pl.col("d").filter(pl.col("gmv") > 0).max()),
    )
    full = pl.DataFrame({"user_id": USERS}).join(f, on="user_id", how="left")
    # 999 = «не было вовсе»: для recency это осмысленнее нуля или пропуска
    return (full.with_columns([pl.col(c).fill_null(999).cast(pl.Int32) for c in ["r_act", "r_gmv"]]
                              + [pl.col(c).fill_null(0) for c in
                                 ["gmv_90", "gmv_180", "ord_days_90", "act_days_30"]])
                .sort("user_id"))


R_B = np.array([0, 1, 3, 7, 14, 30, 60, 120, 10_000])     # границы бинов recency
F_B = np.array([0, 1, 2, 4, 8, 16, 32, 1000])             # дней с покупкой за 90д
G_B = np.array([0, 1, 50, 200, 600, 1500, 4000, 1e12])    # GMV за 90д

def cells(F):
    a = np.digitize(F["r_act"].to_numpy(), R_B[1:-1])
    b = np.digitize(F["ord_days_90"].to_numpy(), F_B[1:-1])
    c = np.digitize(F["gmv_90"].to_numpy(), G_B[1:-1])
    return a * (len(F_B) - 1) * (len(G_B) - 1) + b * (len(G_B) - 1) + c

F_tr, F_va = compact_features(ANCHOR_TRAIN), compact_features(ANCHOR_VAL)
y_tr = make_target(ANCHOR_TRAIN)["y"].to_numpy()
c_tr, c_va = cells(F_tr), cells(F_va)
n_cells = (len(R_B) - 1) * (len(F_B) - 1) * (len(G_B) - 1)

cnt = np.bincount(c_tr, minlength=n_cells).astype("float64")
n_pos = np.bincount(c_tr, weights=(y_tr > 0).astype("float64"), minlength=n_cells)
s_log = np.bincount(c_tr, weights=np.where(y_tr > 0, np.log1p(y_tr), 0.0), minlength=n_cells)

prior_p = (y_tr > 0).mean()
prior_m = np.log1p(y_tr[y_tr > 0]).mean()
K = 30.0                                       # сила притяжения к глобальному среднему
p_cell = (n_pos + K * prior_p) / (cnt + K)
m_cell = (s_log + K * prior_m) / (n_pos + K)
p_hat, m_hat = p_cell[c_va], m_cell[c_va]

print(f"ячеек всего {n_cells}, непустых в обучающем якоре {(cnt>0).sum()}, "
      f"покрытие валидации {np.mean(cnt[c_va] > 0):.3f}")
print(f"обучение: якорь {DAY0+timedelta(days=ANCHOR_TRAIN)} → "
      f"применение: якорь {DAY0+timedelta(days=ANCHOR_VAL)} (окна таргета не пересекаются)")

ячеек всего 392, непустых в обучающем якоре 168, покрытие валидации 1.000
обучение: якорь 2025-12-15 → применение: якорь 2026-01-14 (окна таргета не пересекаются)


In [14]:
variants = {
    "exp(p·m) − 1  — правило под RMSLE":        np.expm1(p_hat * m_hat),
    "p · (exp(m) − 1)  — интуитивная склейка":  p_hat * np.expm1(m_hat),
    "exp(m) − 1  — регрессия без классификатора": np.expm1(m_hat),
    "оптимальная константа":                    np.full_like(y, best_const),
}
res_h = [(k, rmsle(y, v)) for k, v in variants.items()]
for k, s_ in sorted(res_h, key=lambda t: t[1]):
    print(f"{s_:.4f}   {k}")

rs = sorted(res_h, key=lambda t: t[1], reverse=True)
fig = go.Figure(go.Bar(
    y=[k for k, _ in rs], x=[v for _, v in rs], orientation="h",
    marker=dict(color=[STATUS["good"] if "RMSLE" in k else SERIES[0] for k, _ in rs],
                cornerradius=4, line=dict(color=SURFACE, width=1)),
    text=[f"{v:.4f}" for _, v in rs], textposition="outside",
    textfont=dict(color=INK2, size=12.5), cliponaxis=False))
fig.update_xaxes(title="RMSLE (меньше — лучше)", range=[0, max(v for _, v in rs) * 1.2])
fig.update_layout(showlegend=False, hovermode="closest")
show(fig, "Одна модель, четыре способа собрать из неё ответ",
     sub="p и m одинаковы, отличается только склейка",
     w=1000, h=520)

1.7468   exp(p·m) − 1  — правило под RMSLE
2.1207   p · (exp(m) − 1)  — интуитивная склейка
2.2883   оптимальная константа
2.6993   exp(m) − 1  — регрессия без классификатора


*Что это даёт.* Разница между правильной и интуитивной склейкой — **0.37 RMSLE**
при полностью идентичных $p$ и $m$. Это самое дешёвое улучшение во всём соревновании:
одна строка кода, никакого обучения. Заодно видно, что регрессия без классификатора
(третья строка) хуже даже константы — без «шлагбаума» модель предсказывает покупку всем.

### Поправка на передискретизацию

В решениях GA Customer Revenue положительные примеры дублировали вплоть до 60 раз,
чтобы «помочь» классификатору. Копировать это бездумно нельзя: передискретизация
с коэффициентом $w$ сдвигает вероятность, и её нужно вернуть назад:

$$
p \;=\; \frac{p_s}{p_s + (1-p_s)\,w}
$$

где $p_s$ — вероятность, выданная моделью на пересэмплированной выборке.
Без этой поправки вы улучшаете AUC и ухудшаете RMSLE: порядок объектов не меняется,
а калибровка — которая тут и есть метрика — ломается.

### Как проверять калибровку

График ниже — стандартная проверка: разбиваем пользователей на 20 групп по
**предсказанной** вероятности и в каждой смотрим **фактическую** долю купивших.
Идеально откалиброванная модель ложится на диагональ. Отклонение вверх означает,
что модель занижает вероятность, вниз — завышает.

In [15]:
qs = np.unique(np.quantile(p_hat, np.linspace(0, 1, 21)))
b = np.clip(np.digitize(p_hat, qs[1:-1]), 0, len(qs) - 2)
xs_, ys_ = [], []
for k in range(len(qs) - 1):
    m_ = b == k
    if m_.sum() > 200:                       # группы меньше 200 человек не показываем: шум
        xs_.append(p_hat[m_].mean()); ys_.append((y[m_] > 0).mean())
fig = go.Figure()
fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines", name="идеальная калибровка",
                         line=dict(color=AXIS, width=1.5, dash="dash")))
fig.add_trace(go.Scatter(x=xs_, y=ys_, mode="markers+lines", name="биннинг-модель",
                         line=dict(color=SERIES[0], width=2.5),
                         marker=dict(size=10, line=dict(color=SURFACE, width=2)),
                         hovertemplate="прогноз %{x:.3f} → факт %{y:.3f}<extra></extra>"))
fig.update_xaxes(title="предсказанная вероятность покупки", range=[0, 1])
fig.update_yaxes(title="фактическая доля купивших", range=[0, 1])
fig.update_layout(hovermode="closest")
show(fig, "Калибровка вероятности покупки",
     sub="20 групп по предсказанной вероятности",
     w=720, h=640)

---
# 6. Куда вкладывать усилия

## 6.1 Обе части hurdle на одной карте

Мы знаем, что модель состоит из двух частей. Логичный следующий вопрос: какая из них
несёт сигнал? Построим одну и ту же сетку ячеек по двум признакам — **recency**
(насколько давно человек заходил) и **частота покупок** (сколько дней с покупкой
было за 90 дней — классические R и F из RFM-анализа) — и посмотрим в каждой ячейке
на обе величины.

**Почему ячейки с малым числом людей выброшены.** В ячейках, где меньше 50 пользователей,
доля покупок скачет от 0 до 1 на случайности, и карта превращается в шум.
Такие ячейки оставлены пустыми, чтобы глаз не читал в них закономерность.

In [16]:
rb = np.array([0, 1, 3, 7, 14, 30, 60, 120, 10_000])
fb = np.array([0, 1, 2, 4, 8, 16, 32, 400])
ri_ = np.digitize(F_va["r_act"].to_numpy(), rb[1:-1])
fi_ = np.digitize(F_va["ord_days_90"].to_numpy(), fb[1:-1])
Pp = np.full((len(fb) - 1, len(rb) - 1), np.nan)
Ml = np.full_like(Pp, np.nan)
for i in range(len(fb) - 1):
    for j in range(len(rb) - 1):
        m_ = (fi_ == i) & (ri_ == j)
        if m_.sum() >= 50:                            # порог против шума
            yy = y[m_]
            Pp[i, j] = (yy > 0).mean()
            if (yy > 0).any():
                Ml[i, j] = np.log1p(yy[yy > 0]).mean()
rl = [f"{rb[j]}–{rb[j+1] if j < len(rb)-2 else '∞'}" for j in range(len(rb) - 1)]
fl = [f"{fb[i]}–{fb[i+1] if i < len(fb)-2 else '∞'}" for i in range(len(fb) - 1)]

fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.15,
                    subplot_titles=["p(x) — вероятность покупки",
                                    "m(x) — сумма в логарифмах, если купит"])
fig.add_trace(heat(Pp, rl, fl, hover="%{z:.3f}"), row=1, col=1)
fig.add_trace(heat(Ml, rl, fl, hover="%{z:.2f}"), row=1, col=2)
fig.data[0].colorbar = dict(thickness=11, outlinewidth=0, len=0.8, x=0.415,
                            tickfont=dict(color=MUTED, size=10))
fig.data[1].colorbar = dict(thickness=11, outlinewidth=0, len=0.8, x=1.005,
                            tickfont=dict(color=MUTED, size=10))
for c in (1, 2):
    fig.update_xaxes(title="дней с последней активности", row=1, col=c)
fig.update_yaxes(title="дней с покупкой за 90 дней", row=1, col=1)
sub_titles(fig)
show(fig, "Две части hurdle на карте recency × частота",
     sub="сравнивать диапазоны, а не цвета",
     w=1060, h=620)
print(f"p(x): от {np.nanmin(Pp):.3f} до {np.nanmax(Pp):.3f} — разброс {np.nanmax(Pp)/np.nanmin(Pp):.1f}×")
print(f"m(x): от {np.nanmin(Ml):.2f} до {np.nanmax(Ml):.2f} — разброс {np.nanmax(Ml)/np.nanmin(Ml):.2f}×")

p(x): от 0.124 до 0.998 — разброс 8.0×
m(x): от 3.39 до 6.19 — разброс 1.83×


*Что это даёт.* По одной и той же сетке $p(x)$ пробегает **восьмикратный** диапазон
(от 0.12 до почти 1.0), а $m(x)$ — менее чем **двукратный** (3.4 → 6.2).
Два простейших признака почти полностью определяют, купит ли человек, и почти ничего
не говорят о том, на сколько.

Практический вывод прямой: ёмкость модели, время разработки и признаки — в классификатор.
Регрессия суммы гораздо более шумная задача, и вкладываться в неё стоит в последнюю очередь.

Отдельно стоит отметить, почему recency здесь работает так сильно. В задаче
Google Analytics Customer Revenue между концом истории и окном таргета был **cooling
period в 46 дней** — за полтора месяца свежесть последнего визита успевала устареть.
У нас окно таргета начинается **на следующий день** после последнего наблюдения.
Поэтому переносить оттуда выводы о важности признаков напрямую нельзя: у нас recency
сильнее, чем было там.

## 6.2 Сколько стоит регрессия к среднему

Прошлый GMV — очевидный кандидат в прогноз. Но переносить его в будущее один к одному
нельзя: у кого месяц был удачный, следующий будет скромнее, и наоборот.
Измерим, насколько именно нужно «поджимать», подобрав показатель $a$:

$$\hat y = \exp\big(a \cdot \log(1 + x)\big) - 1, \qquad x = \text{GMV за прошлые } W \text{ дней}$$

При $a = 1$ это буквальный перенос прошлого в будущее. При $a < 1$ прогноз сжимается
к нулю тем сильнее, чем крупнее был прошлый GMV, — то есть моделируется возврат к среднему.

In [17]:
g30, g60, g90, g365 = (past_sum(ANCHOR_VAL, w) for w in (30, 60, 90, 365))
alphas = np.linspace(0.0, 1.3, 40)
fig = go.Figure()
best = {}
for i, (nm, base) in enumerate([("прошлые 30 дней", g30), ("прошлые 90 дней / 3", g90 / 3),
                                ("год × 30/365", g365 * 30 / 365)]):
    c_ = [rmsle(y, np.expm1(a * np.log1p(base))) for a in alphas]
    line(fig, alphas, c_, nm, slot=i)
    j = int(np.argmin(c_)); best[nm] = (alphas[j], c_[j])
    fig.add_trace(go.Scatter(x=[alphas[j]], y=[c_[j]], mode="markers+text",
                             marker=dict(color=SERIES[i], size=11,
                                         line=dict(color=SURFACE, width=2)),
                             text=[f" a={alphas[j]:.2f}"], textposition="top right",
                             textfont=dict(color=INK2, size=11), showlegend=False))
fig.add_vline(x=1.0, line=dict(color=AXIS, width=1.5, dash="dash"),
              annotation_text="перенос один к одному", annotation_position="bottom right",
              annotation_font=dict(color=MUTED, size=11))
fig.update_xaxes(title="показатель сжатия a")
fig.update_yaxes(title="RMSLE")
fig.update_layout(hovermode="x")
show(fig, "Оптимальное сжатие прошлого GMV",
     w=920, h=580)
for nm, (a_, s_) in best.items():
    print(f"{nm:<24} a* = {a_:.2f}   RMSLE = {s_:.4f}")

прошлые 30 дней          a* = 0.73   RMSLE = 2.0099
прошлые 90 дней / 3      a* = 0.73   RMSLE = 1.8348
год × 30/365             a* = 0.73   RMSLE = 1.8238


*Что это даёт.* Минимум для всех трёх окон — около $a \approx 0.73$, устойчиво меньше
единицы. Это тот эффект, который модель должна выучить сама; если она его не выучила
(например, обучалась в исходной шкале), прогноз будет систематически завышен на активных
пользователях. Заодно видно, что длинное окно (год) даёт лучший результат, чем короткое, —
месяц слишком шумный, чтобы быть надёжной оценкой уровня пользователя.

## 6.3 Планка

Соберём всё вместе: что даёт каждый подход на валидационном якоре.

Вторая колонка — та же метрика, но на **популяции с фильтром отбора** из раздела 3.
Обе колонки полезны: первая сравнима с чужими публичными цифрами, вторая честнее
отражает то, что будет на боевом якоре.

In [18]:
baselines = {
    "все нули":                              np.zeros_like(y),
    "среднее y (наивно)":                    np.full_like(y, y.mean()),
    "оптимальная константа c*":              np.full_like(y, best_const),
    "прошлые 30 дней (= sample_submit)":     g30,
    "прошлые 90 дней / 3":                   g90 / 3,
    "год × 30/365":                  g365 * 30 / 365,
    "сжатие: exp(0.70·log1p(90д/3)) − 1":    np.expm1(0.70 * np.log1p(g90 / 3)),
    "биннинг-hurdle: exp(p·m) − 1":          np.expm1(p_hat * m_hat),
}
sel_val = selected_users(ANCHOR_VAL)
mask_sel = np.isin(USERS.to_numpy(), sel_val.to_numpy())
rows = [(k, rmsle(y, v), rmsle(y[mask_sel], np.asarray(v)[mask_sel])) for k, v in baselines.items()]
rows.sort(key=lambda t: t[1])
print(f"популяция с фильтром: {mask_sel.sum():,} из {N_USERS:,} ({mask_sel.mean():.1%})\n")
print(f"{'подход':<38} {'все 250K':>10} {'с фильтром':>12}")
for k, a_, b_ in rows:
    print(f"{k:<38} {a_:>10.4f} {b_:>12.4f}")

rr = rows[::-1]
fig = go.Figure(go.Bar(
    y=[k for k, _, _ in rr], x=[v for _, v, _ in rr], orientation="h",
    marker=dict(color=[STATUS["good"] if "hurdle" in k else SERIES[0] for k, _, _ in rr],
                cornerradius=4, line=dict(color=SURFACE, width=1)),
    text=[f"{v:.4f}" for _, v, _ in rr], textposition="outside",
    textfont=dict(color=INK2, size=11.5), cliponaxis=False))
fig.update_xaxes(title="RMSLE на валидационном якоре", range=[0, max(v for _, v, _ in rr) * 1.18])
fig.update_layout(showlegend=False, hovermode="closest")
show(fig, "Планка, которую должна перебить модель",
     w=1000, h=620)

популяция с фильтром: 233,152 из 250,000 (93.3%)

подход                                   все 250K   с фильтром
биннинг-hurdle: exp(p·m) − 1               1.7468       1.7596
сжатие: exp(0.70·log1p(90д/3)) − 1         1.8361       1.8488
год × 30/365                               2.0640       2.0822
прошлые 90 дней / 3                        2.0936       2.1100
прошлые 30 дней (= sample_submit)          2.1951       2.2109
оптимальная константа c*                   2.2883       2.2966
среднее y (наивно)                         3.1750       3.1142
все нули                                   3.2036       3.2756


---
# 7. Причинные проверки

Соревнование чисто предсказательное, и причинность нужна здесь не ради лидерборда.
Она отвечает на два вопроса, которые иначе легко проглядеть: какие сигналы действительно
ведут к покупке, а какие просто маркируют активного человека, — и не строим ли мы модель
на признаке, который на самом деле следствие, а не причина.

Важная оговорка на весь раздел: рандомизации нет, это не A/B-тест. Все оценки верны
под допущением *selection on observables* — что мы учли все существенные различия
между группами. Корректная формулировка вывода: «разница между сопоставимыми
по наблюдаемым признакам группами», а не «эффект вмешательства».

## 7.1 Промо переносят спрос, а не создают его

**Зачем это нам.** Если в окно признаков попал промо-день, суммарный GMV за 30 дней
окажется завышенным относительно обычного уровня пользователя. Вопрос в том, окупается
ли всплеск последующим ростом, или люди просто купили раньше то, что купили бы позже.
От ответа зависит, нужно ли чистить признак `gmv_30d` от календарных эффектов.

**Как измеряем.** Берём четыре крупнейших пика GMV, разнесённых минимум на 21 день,
и смотрим динамику вокруг каждого. Базовая линия — медиана GMV **того же дня недели**
за 8 недель до события: так снимается недельная сезонность, из-за которой воскресенье
нельзя сравнивать со средой. Если после пика кривая уходит ниже единицы —
спрос был передвинут во времени.

In [19]:
daily = (df.group_by("d").agg(gmv=pl.col("gmv").sum(), dau=pl.len())
           .sort("d").with_columns(dow=((pl.col("d") + 2) % 7).cast(pl.Int8)))
peaks = daily.sort("gmv", descending=True).head(40)
selp, used = [], []
for r_ in peaks.iter_rows(named=True):
    if all(abs(r_["d"] - u) > 21 for u in used):     # пики не ближе 21 дня друг к другу
        selp.append(r_); used.append(r_["d"])
    if len(selp) == 4:
        break
selp = sorted(selp, key=lambda r_: r_["d"])

fig = go.Figure()
for i, r_ in enumerate(selp):
    a = r_["d"]
    seg = daily.filter(pl.col("d").is_between(max(0, a - 21), min(N_DAYS - 1, a + 21))).sort("d")
    bl = (daily.filter(pl.col("d").is_between(a - 56, a - 8))     # 8 недель до, с зазором
               .group_by("dow").agg(b=pl.col("gmv").median()))
    seg = seg.join(bl, on="dow", how="left")
    line(fig, (seg["d"] - a).to_numpy(), (seg["gmv"] / seg["b"]).to_numpy(),
         (DAY0 + timedelta(days=a)).strftime("%Y-%m-%d"), slot=i, width=2.2)
fig.add_hline(y=1.0, line=dict(color=AXIS, width=1.5, dash="dash"))
fig.add_vline(x=0, line=dict(color=INK, width=1.5))
fig.update_xaxes(title="дней от пикового дня")
fig.update_yaxes(title="GMV к базовой линии того же дня недели", type="log")
fig.update_layout(hovermode="x")
show(fig, "Event study вокруг пиковых дней",
     sub="1.0 — обычный уровень того же дня недели",
     w=960, h=600)
for r_ in selp:
    a = r_["d"]
    pre = daily.filter(pl.col("d").is_between(a - 14, a - 1))["gmv"].sum()
    post = daily.filter(pl.col("d").is_between(a + 1, a + 14))["gmv"].sum()
    flag = "  ← спрос перенесён" if post / pre < 0.9 else ""
    print(f"{(DAY0+timedelta(days=a)).isoformat()}: 14 дней после / 14 дней до = {post/pre:.3f}{flag}")

2025-10-15: 14 дней после / 14 дней до = 0.985
2025-11-11: 14 дней после / 14 дней до = 1.116
2025-12-03: 14 дней после / 14 дней до = 1.142
2025-12-27: 14 дней после / 14 дней до = 0.715  ← спрос перенесён


*Что это даёт.* Три из четырёх пиков нейтральны или дают рост, но новогодний
показывает явный перенос: две недели после дают лишь 72 % от двух недель до.
Значит промо-эффект не универсален, и для якорей, чьё окно признаков захватывает
конец декабря, `gmv_30d` завышен. Практически: добавить календарные признаки якоря
либо не строить признаки только на коротком окне.

## 7.2 Конфаундинг может перевернуть знак

**Вопрос.** Пользователи, которые больше пользуются Поиском, приносят больше GMV?
Наивное сравнение бесполезно: кто вообще активнее, тот и ищет больше, и покупает больше.
Общая активность здесь — конфаундер, то есть общая причина обеих величин.

**Как исправляем.** Строим propensity score — вероятность попасть в «поисковую» группу,
предсказанную по ковариатам активности (стаж, recency, число активных дней, число дней
с покупкой, прошлый GMV, корзины, запросы). Затем сравниваем людей с **одинаковым**
скором: IPW взвешивает наблюдения обратно вероятности попадания в свою группу,
матчинг подбирает каждому «поисковому» пользователю ближайшего «каталожного».

**Почему такая подвыборка.** Оставляем тех, у кого за 90 дней минимум 5 активных дней
и был хотя бы какой-то GMV. Причина проста: у человека с двумя визитами доля каналов
не измерима — она равна 0, 0.5 или 1 просто по случайности, — а без покупок сравнение
не о чем. Это фильтр на измеримость признака, а не отбор под нужный результат.

**Почему разрез квантильный.** Первая попытка делила по порогу «≥0.75 Поиска против
≤0.25» и дала 99.7 % против 0.3 %: чистого «каталожного» сегмента в данных
попросту нет, медианная доля поисковых дней ≈ 0.90. Контрольной группы не существует,
и оценка превращается в экстраполяцию. Поэтому сравниваем **верхний квартиль
по доле Поиска против нижнего** — реальный контраст внутри существующей популяции.

In [20]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

conf_src = df.filter(pl.col("d").is_between(ANCHOR_VAL - 89, ANCHOR_VAL)).group_by("user_id").agg(
    sd=pl.col("search").sum(), cd=pl.col("cat").sum(),
    act90=pl.len(), ord_days=(pl.col("to_ord") > 0).sum(),
    gmv90=pl.col("gmv").sum(), cart90=pl.col("to_cart").sum(),
    srch90=pl.col("searches").sum(), r_act=(ANCHOR_VAL - pl.col("d").max()),
)
share_all = (conf_src["sd"] / (conf_src["sd"] + conf_src["cd"] + 1e-9)).to_numpy()
B = (conf_src.join(up.select(["user_id", "first_d", "n_days"]), on="user_id")
             .with_columns(share=pl.col("sd") / (pl.col("sd") + pl.col("cd") + 1e-9),
                           tenure=(ANCHOR_VAL - pl.col("first_d")))
             .filter((pl.col("act90") >= 5) & (pl.col("gmv90") > 0)))   # фильтр измеримости
q_lo, q_hi = np.quantile(B["share"].to_numpy(), [0.25, 0.75])
print(f"доля поисковых дней по всей базе: p25={np.quantile(share_all,.25):.3f}, "
      f"медиана={np.median(share_all):.3f}, p75={np.quantile(share_all,.75):.3f}")
print(f"→ «каталожных» пользователей как отдельного сегмента не существует, режем по квартилям")
B = B.with_columns(W=pl.when(pl.col("share") >= q_hi).then(1)
                     .when(pl.col("share") <= q_lo).then(0).otherwise(None)).drop_nulls("W").sort("user_id")

CONF = ["tenure", "n_days", "r_act", "act90", "ord_days", "gmv90", "cart90", "srch90"]
Wt = B["W"].to_numpy().astype(int)
Xraw = B.select(CONF).to_numpy().astype("float64")
Xc = StandardScaler().fit_transform(np.sign(Xraw) * np.log1p(np.abs(Xraw)))
lyb = np.log1p(tgt.join(B.select("user_id"), on="user_id", how="semi").sort("user_id")["y"].to_numpy())
print(f"\nвыборка после фильтра измеримости: {len(Wt):,}, из них верхний квартиль {Wt.mean():.1%}")

ps = np.clip(LogisticRegression(max_iter=1000).fit(Xc, Wt).predict_proba(Xc)[:, 1], 1e-3, 1 - 1e-3)
w_att = np.where(Wt == 1, 1.0, ps / (1 - ps))
naive = lyb[Wt == 1].mean() - lyb[Wt == 0].mean()
ipw = (np.average(lyb[Wt == 1], weights=w_att[Wt == 1])
       - np.average(lyb[Wt == 0], weights=w_att[Wt == 0]))
tr, ct = np.where(Wt == 1)[0], np.where(Wt == 0)[0]
lg = np.log(ps / (1 - ps))
dist, ind = NearestNeighbors(n_neighbors=1).fit(lg[ct].reshape(-1, 1)).kneighbors(lg[tr].reshape(-1, 1))
ok = dist.ravel() < 0.05                                   # caliper: пары только близкие
matched = lyb[tr[ok]] - lyb[ct[ind.ravel()[ok]]]
bs = np.array([matched[RNG.integers(0, len(matched), len(matched))].mean() for _ in range(300)])
ci = np.percentile(bs, [2.5, 97.5])
print(f"\nнаивная разница : {naive:+.4f}")
print(f"IPW (ATT)       : {ipw:+.4f}")
print(f"матчинг 1:1     : {matched.mean():+.4f}  95% CI [{ci[0]:+.4f}, {ci[1]:+.4f}]")

est = [("наивная разница", naive), ("IPW", ipw), ("матчинг 1:1", matched.mean())]
fig = go.Figure(go.Bar(
    x=[e[0] for e in est], y=[e[1] for e in est],
    marker=dict(color=[STATUS["critical"], SERIES[0], SERIES[2]], cornerradius=4,
                line=dict(color=SURFACE, width=1)),
    error_y=dict(type="data", symmetric=False, array=[0, 0, ci[1] - matched.mean()],
                 arrayminus=[0, 0, matched.mean() - ci[0]], color=INK2, thickness=1.5, width=6),
    text=[f"{e[1]:+.3f}" for e in est], textposition="outside",
    textfont=dict(color=INK2, size=12.5), cliponaxis=False))
fig.add_hline(y=0, line=dict(color=AXIS, width=1.5))
fig.update_yaxes(title="разница в среднем log1p(GMV за 30 дней)")
fig.update_layout(showlegend=False, hovermode="closest")
show(fig, "Эффект высокой интенсивности Поиска на будущий GMV",
     sub="до и после поправки на активность",
     w=820, h=580)

доля поисковых дней по всей базе: p25=0.800, медиана=0.904, p75=0.979
→ «каталожных» пользователей как отдельного сегмента не существует, режем по квартилям

выборка после фильтра измеримости: 93,468, из них верхний квартиль 50.0%

наивная разница : -0.2549
IPW (ATT)       : +0.1332
матчинг 1:1     : +0.0055  95% CI [-0.0207, +0.0359]


*Что это даёт.* Наивно «поисковые» пользователи выглядят заметно **хуже** — разница
отрицательная. После поправки на активность знак меняется: IPW даёт слабо положительную
оценку, матчинг — практически ноль с доверительным интервалом, накрывающим ноль.
Весь видимый «отрицательный эффект Поиска» был конфаундингом.

Практический смысл: доля канала сама по себе ничего не «делает» с GMV — она маркирует
тип пользователя. В модель её класть можно (как признак типа), но интерпретировать
как рычаг нельзя. Расхождение между IPW и матчингом — честный признак того, что точечная
оценка здесь неустойчива, и выводы стоит делать только про знак и порядок величины.

## 7.3 Что добавляет информацию сверх истории покупок

**Вопрос.** Стоит ли вообще вкладываться в признаки по корзинам и поисковым запросам,
или они просто повторяют историю покупок другими словами?

**Как измеряем.** Собираем недельную панель: для каждого пользователя ряд из
`log1p` недельных сумм. Сравниваем вложенные регрессии — предсказание GMV недели $t$
по четырём предыдущим неделям GMV против той же модели плюс лаги корзин или запросов.
Прирост $R^2$ и есть добавленная информация.

**Почему по неделям, а не по дням.** Дневной ряд слишком разрежен (раздел 2.1):
у большинства пользователей в конкретный день просто нет строки, и лаги были бы
почти сплошными нулями. Неделя — минимальная гранулярность, на которой ряд осмысленный.

**Почему вычитаем среднее по пользователю.** Это фиксированные эффекты: без них
регрессия просто выучила бы, что «активные пользователи покупают много» — тот же
конфаундер, что и в разделе 7.2. Вычитая среднее каждого пользователя, мы оставляем
только **внутреннюю** динамику: у этого конкретного человека неделя была лучше или
хуже его собственной нормы. Именно это нас и интересует.

**Почему 30 000 пользователей.** Плотная матрица на всю базу — это 250 000 × 59 недель
на четыре величины; выборка в 30 000 даёт ту же точность оценки $R^2$ при вчетверо
меньшем расходе памяти.

In [21]:
GR_N = 30_000
gids = np.sort(RNG.choice(USERS.to_numpy(), size=GR_N, replace=False))
wk = (df.filter(pl.col("user_id").is_in(gids))
        .with_columns(w=(pl.col("d") // 7).cast(pl.Int16))
        .group_by(["user_id", "w"]).agg(gmv=pl.col("gmv").sum(), cart=pl.col("to_cart").sum(),
                                        srch=pl.col("searches").sum(), act=pl.len()))
n_w = int(df["d"].max() // 7) + 1
rowi = {u: i for i, u in enumerate(gids)}
ri2 = np.array([rowi[u] for u in wk["user_id"].to_numpy()]); ci2 = wk["w"].to_numpy().astype(int)
P_ = {}
for nm in ["gmv", "cart", "srch", "act"]:
    Mx = np.zeros((GR_N, n_w), dtype="float32"); Mx[ri2, ci2] = wk[nm].to_numpy()
    P_[nm] = np.log1p(Mx)

L = 4                                                   # четыре недельных лага
demean = lambda a: a - a.mean(1, keepdims=True)         # фиксированные эффекты пользователя
Y = demean(P_["gmv"][:, L:]).ravel()
lags = lambda nm: [P_[nm][:, L - k: -k if k else None] for k in range(1, L + 1)]
def r2(blocks):
    Xd = np.column_stack([demean(b).ravel() for b in blocks] + [np.ones(len(Y))])
    beta, *_ = np.linalg.lstsq(Xd, Y, rcond=None)
    return 1 - (Y - Xd @ beta).var() / Y.var()

models = {"только лаги GMV": lags("gmv"),
          "+ лаги корзин": lags("gmv") + lags("cart"),
          "+ лаги запросов": lags("gmv") + lags("srch"),
          "+ лаги активных дней": lags("gmv") + lags("act"),
          "всё вместе": lags("gmv") + lags("cart") + lags("srch") + lags("act")}
sc_ = {k: r2(v) for k, v in models.items()}
b0 = sc_["только лаги GMV"]
for k, v in sc_.items():
    print(f"R² = {v:.4f}  (прирост {v-b0:+.4f})   {k}")
fig = go.Figure(bars(list(sc_), [sc_[k] - b0 for k in sc_], slot=0,
                     text=[f"{sc_[k]-b0:+.4f}" for k in sc_]))
fig.update_yaxes(title="прирост R² к модели «только лаги GMV»")
fig.update_layout(showlegend=False, hovermode="closest")
show(fig, "Не-GMV-сигналы не дублируют историю покупок",
     sub=f"{GR_N:,} пользователей, внутренняя вариация",
     w=920, h=560)

R² = 0.0163  (прирост +0.0000)   только лаги GMV
R² = 0.0245  (прирост +0.0082)   + лаги корзин
R² = 0.0268  (прирост +0.0105)   + лаги запросов
R² = 0.0251  (прирост +0.0088)   + лаги активных дней
R² = 0.0284  (прирост +0.0121)   всё вместе


*Что это даёт.* База на лагах GMV объясняет 1.6 % внутрипользовательской вариации —
мало, но так и должно быть: недельное поведение человека почти непредсказуемо.
Важно другое: добавление корзин или запросов даёт прирост **того же порядка, что и вся
база**. То есть поведенческие сигналы несут информацию о будущем GMV сверх истории
покупок, а не повторяют её. Вкладываться в признаки по корзинам, запросам
и активности — оправдано.

Оговорка, без которой вывод неверен: это Granger-причинность, то есть утверждение
о предсказуемости, а не о механизме. Общий ненаблюдаемый драйвер (сезон, возникшая
потребность, рекламная кампания) дал бы ту же картину. Фиксированные эффекты снимают
устойчивые различия между людьми, но не динамические.

## 7.4 Что из этого следует про конкретные признаки

Причинные проверки дали качественные утверждения. Переведём их в число: возьмём
базовую модель из трёх признаков (recency, число активных дней за 90 дней, GMV за 90 дней)
и для каждого кандидата посмотрим, **сколько он добавляет сверх базы** и **не меняет ли
знак** при переходе от одиночной корреляции к модели.

Смена знака — это ровно тот эффект, который мы видели в разделе 7.2: признак сам по себе
говорит одно, а при контроле активности — противоположное. Такой признак опасен: модель
без хороших контролей выучит по нему неверное направление.

In [22]:
A_ = ANCHOR_VAL
h_ = df.filter(pl.col("d").is_between(A_ - 364, A_))
Fx = h_.group_by("user_id").agg(
    r_act=(A_ - pl.col("d").max()),
    r_gmv=(A_ - pl.col("d").filter(pl.col("gmv") > 0).max()),
    act90=(pl.col("d") > A_ - 90).sum(),
    gmv90=pl.col("gmv").filter(pl.col("d") > A_ - 90).sum(),
    gmv30=pl.col("gmv").filter(pl.col("d") > A_ - 30).sum(),
    gmv365=pl.col("gmv").sum(),
    gmv_prev30=pl.col("gmv").filter(pl.col("d").is_between(A_ - 59, A_ - 30)).sum(),
    cart90=pl.col("to_cart").filter(pl.col("d") > A_ - 90).sum(),
    srch90=pl.col("searches").filter(pl.col("d") > A_ - 90).sum(),
    ordd90=((pl.col("d") > A_ - 90) & (pl.col("to_ord") > 0)).sum(),
    ord90=pl.col("to_ord").filter(pl.col("d") > A_ - 90).sum(),
    sd=pl.col("search").filter(pl.col("d") > A_ - 90).sum(),
    cd=pl.col("cat").filter(pl.col("d") > A_ - 90).sum(),
    wknd=((pl.col("d") > A_ - 90) & (pl.col("dow") >= 5)).sum(),
    phant=((pl.col("d") > A_ - 90) & (pl.col("search") == 0) & (pl.col("cat") == 0)).sum(),
    tenure=(A_ - pl.col("d").min()))
Fx = pl.DataFrame({"user_id": USERS}).join(Fx, on="user_id", how="left").fill_null(0).sort("user_id")
Fx = Fx.filter(pl.Series(mask_sel))                       # популяция, сопоставимая с тестом
zt = np.log1p(y[mask_sel])
cx = Fx.to_dict(as_series=False)
lg = lambda a: np.log1p(np.asarray(a, dtype="float64"))

base = np.column_stack([lg(cx["r_act"]), lg(cx["act90"]), lg(cx["gmv90"]), np.ones(len(zt))])
cand = {
    "дней с покупкой за 90д": lg(cx["ordd90"]),
    "средний чек за 90д":     lg(np.array(cx["gmv90"]) / (np.array(cx["ord90"]) + 1e-6)),
    "GMV за 30д":             lg(cx["gmv30"]),
    "GMV за 365д":            lg(cx["gmv365"]),
    "корзины за 90д":         lg(cx["cart90"]),
    "recency покупки":        lg(cx["r_gmv"]),
    "запросы за 90д":         lg(cx["srch90"]),
    "доля фантомных дней":    np.array(cx["phant"]) / (np.array(cx["act90"]) + 1e-6),
    "стаж":                   lg(cx["tenure"]),
    "тренд GMV 30 / 60":      lg(cx["gmv30"]) - lg(cx["gmv_prev30"]),
    "доля Поиска за 90д":     np.array(cx["sd"]) / (np.array(cx["sd"]) + np.array(cx["cd"]) + 1e-9),
    "доля выходных за 90д":   np.array(cx["wknd"]) / (np.array(cx["act90"]) + 1e-6),
}

def fit(X):
    b, *_ = np.linalg.lstsq(X, zt, rcond=None)
    return 1 - (zt - X @ b).var() / zt.var(), b

r2_base, _ = fit(base)
rows_f = []
for k, v in cand.items():
    v = np.nan_to_num(np.asarray(v, dtype="float64"), posinf=0, neginf=0)
    raw = float(np.corrcoef(v, zt)[0, 1])
    r2_f, b = fit(np.column_stack([base, v]))
    rows_f.append((k, raw, r2_f - r2_base, float(b[-1]), raw * b[-1] < 0 and abs(raw) > 0.05))

print(f"популяция {len(zt):,} · база (recency + активность + GMV за 90д): R² = {r2_base:.4f}\n")
def verdict_of(raw, dr, b, flip):
    if flip:
        return "ПЕРЕВОРОТ знака"
    if abs(raw) < 0.05 and abs(b) > 0.15:      # сам по себе молчит, в модели говорит
        return "работает только с контролями"
    return ("несёт своё" if dr > 0.010 else
            "умеренный вклад" if dr > 0.002 else "балласт")

print(f"{'признак':<24}{'сырая r':>9}{'ΔR²':>9}{'коэф.':>9}   вывод")
for k, raw, dr, b, flip in sorted(rows_f, key=lambda t: -t[2]):
    print(f"{k:<24}{raw:>+9.3f}{dr:>9.4f}{b:>+9.3f}   {verdict_of(raw, dr, b, flip)}")

популяция 233,152 · база (recency + активность + GMV за 90д): R² = 0.3653

признак                   сырая r      ΔR²    коэф.   вывод
средний чек за 90д         +0.420   0.0528   -0.883   ПЕРЕВОРОТ знака
дней с покупкой за 90д     +0.638   0.0459   +1.311   несёт своё
GMV за 30д                 +0.552   0.0183   +0.212   несёт своё
GMV за 365д                +0.563   0.0158   +0.212   несёт своё
корзины за 90д             +0.504   0.0057   +0.216   умеренный вклад
recency покупки            -0.195   0.0055   -0.108   умеренный вклад
запросы за 90д             +0.460   0.0035   +0.221   умеренный вклад
доля фантомных дней        -0.132   0.0015   -0.527   балласт
стаж                       +0.180   0.0010   +0.244   балласт
тренд GMV 30 / 60          +0.023   0.0006   +0.026   балласт
доля Поиска за 90д         -0.007   0.0003   +0.261   работает только с контролями
доля выходных за 90д       -0.015   0.0000   -0.057   балласт


In [23]:
rf = sorted(rows_f, key=lambda t: t[2])
fig = go.Figure(go.Bar(
    y=[r[0] for r in rf], x=[r[2] for r in rf], orientation="h",
    marker=dict(color=[STATUS["warning"] if r[4] else
                       (SERIES[0] if r[2] > 0.002 else AXIS) for r in rf],
                cornerradius=4, line=dict(color=SURFACE, width=1)),
    text=[f"{r[2]:.4f}" + ("  ⚠ знак" if r[4] else "") for r in rf],
    textposition="outside", textfont=dict(color=INK2, size=11), cliponaxis=False))
fig.update_xaxes(title="прирост R² сверх базовых трёх признаков",
                 range=[0, max(r[2] for r in rf) * 1.35])
fig.update_layout(showlegend=False, hovermode="closest")
show(fig, "Что признак добавляет сверх recency, активности и GMV",
     sub="жёлтый — знак меняется при контроле; серый — вклад в пределах шума",
     w=980, h=620)

### Разбор по группам

**Несут собственную информацию — брать обязательно.**
`дней с покупкой за 90д` даёт наибольший прирост среди «честных» признаков:
частота покупок — это не то же самое, что их сумма, и именно она управляет
«шлагбаумом» (раздел 6.1). `GMV за 30д` и `GMV за 365д` добавляют примерно поровну,
хотя между собой сильно скоррелированы: короткое окно ловит текущее состояние,
длинное — устойчивый уровень.

**Меняют знак при контроле — брать можно, интерпретировать нельзя.**
`средний чек за 90д` даёт **самый большой прирост** из всех кандидатов, и при этом
его коэффициент в модели **отрицательный**, тогда как одиночная корреляция
уверенно положительная. Объяснение простое, но заметить его без такой проверки трудно:
при фиксированных общем GMV и активности высокий средний чек означает, что покупок
было **мало** — одна крупная вместо нескольких обычных. А редко покупающий пользователь
хуже предсказуем и в среднем принесёт меньше. То есть признак работает не как
«богатый клиент», а как «редкие крупные покупки», и его вклад целиком зависит
от наличия рядом контролей по объёму и активности.

`доля Поиска за 90д` — тот же случай, что в разделе 7.2, только теперь в числах:
сырая корреляция практически нулевая (−0.007), собственный вклад мизерный (ΔR² = 0.0003),
а коэффициент в модели положительный. Признак не бесполезен, но работает исключительно
как поправка к активности, и любые выводы вида «Поиск приносит больше денег»
по нему делать нельзя.

**Почти дублируют базу.** `корзины` и `запросы` за 90 дней добавляют мало —
как ещё одна 90-дневная сумма они в основном повторяют объём активности.
Но это **не противоречит** разделу 7.3: там они давали прирост, сопоставимый
со всей базой, потому что вопрос был другой — о **внутренней динамике** пользователя
от недели к неделе, а не о его среднем уровне. Отсюда практический вывод:
корзины и запросы стоит подавать как **лаги и тренды**, а не как ещё одну общую сумму.

**Балласт.** `доля выходных` (ΔR² = 0.0000), `тренд GMV 30/60` и `стаж` не добавляют
почти ничего. Тренд особенно показателен: как только в модели есть уровни за оба окна,
их отношение не несёт новой информации — линейная модель выведет его сама.

---
# 8. Схема нарезки: якоря, окна и аномальные месяцы

Теперь соберём разрозненные факты в конкретный рецепт: как резать историю на обучающие
примеры для двухчастной модели, и что делать с месяцами, которые выбиваются из общего ряда.

## 8.1 Базовая нарезка

Один обучающий пример — это пара «пользователь, якорь». Для якоря $T$:

| Компонент | Диапазон | Назначение |
|---|---|---|
| Окно признаков | $[T-167,\; T]$ | 168 дней, как у победителя GA Customer Revenue |
| Таргет | $[T+1,\; T+30]$ | то, что предсказываем |
| Популяция | `selected_users(T)` | активность в каждом из трёх блоков $[T-89,T-60]$, $[T-59,T-30]$, $[T-29,T]$ |

Якоря берём с шагом 30 дней назад от валидационного:

$$T_k = \text{2026-01-14} - 30k, \qquad k = 0, 1, \dots, K$$

$k=0$ — валидация, $k \ge 1$ — обучение. Шаг 30 означает, что окна признаков соседних
якорей перекрываются на 138 дней из 168 — это нормально: перекрываются **признаки**,
а таргет-окна не пересекаются, значит утечки ответа нет. Полностью непересекающихся
окон по 168 дней в нашей истории помещается всего два, и обучаться на них было бы
расточительством.

Ограничение снизу: якорю нужно 168 дней признаков плюс 90 дней на фильтр отбора,
поэтому самый ранний осмысленный якорь — примерно середина июня 2025.

## 8.2 Проблема: окна неравноценны

Тридцатидневные окна отличаются не только составом пользователей, но и **уровнем спроса**.
Ноябрь с распродажами и январь после праздников — это разные вселенные, и модель,
обученная на смеси, будет предсказывать некоторое среднее по больнице.

Измерим. Разложим уровень каждого окна на общий тренд (он же вбирает эффект
взросления панели) и отклонение от него.

In [24]:
daily_g = df.group_by("d").agg(g=pl.col("gmv").sum()).sort("d")["g"].to_numpy()
ends = list(range(N_DAYS - 1, 29, -30))[::-1]
lvl = np.array([daily_g[e - 29:e + 1].sum() for e in ends])
llv = np.log(lvl)
xx = np.arange(len(lvl))
coef = np.polyfit(xx, llv, 1)
resid = llv - np.polyval(coef, xx)
labels = [f"{(DAY0+timedelta(days=e-29)).strftime('%d.%m')}–{(DAY0+timedelta(days=e)).strftime('%d.%m.%y')}"
          for e in ends]
print(f"средний рост окна к окну: {100*(np.exp(coef[0])-1):+.1f}% "
      f"(тренд площадки плюс эффект отбора)\n")
for l_, v, rres in zip(labels, lvl, resid):
    flag = "  ← провал" if rres < -0.12 else ("  ← всплеск" if rres > 0.06 else "")
    print(f"  {l_:<20} GMV {v/1e6:6.2f} млн   отклонение от тренда {100*rres:+6.1f}%{flag}")

cols = [STATUS["critical"] if r < -0.12 else (STATUS["good"] if r > 0.06 else SERIES[0])
        for r in resid]
fig = go.Figure(go.Bar(x=labels, y=100 * resid,
                       marker=dict(color=cols, cornerradius=4,
                                   line=dict(color=SURFACE, width=1))))
fig.add_hline(y=0, line=dict(color=AXIS, width=1.5))
fig.update_yaxes(title="отклонение уровня окна от тренда, %")
fig.update_xaxes(tickangle=-45)
fig.update_layout(showlegend=False, hovermode="closest")
show(fig, "Не все 30-дневные окна одинаково полезны",
     sub="красное — сезонный провал, зелёное — распродажи",
     w=1000, h=580)

средний рост окна к окну: +4.2% (тренд площадки плюс эффект отбора)

  20.01–18.02.25       GMV  14.65 млн   отклонение от тренда   -6.5%
  19.02–20.03.25       GMV  16.75 млн   отклонение от тренда   +2.9%
  21.03–19.04.25       GMV  16.71 млн   отклонение от тренда   -1.4%
  20.04–19.05.25       GMV  16.92 млн   отклонение от тренда   -4.3%
  20.05–18.06.25       GMV  18.41 млн   отклонение от тренда   +0.1%
  19.06–18.07.25       GMV  19.23 млн   отклонение от тренда   +0.3%
  19.07–17.08.25       GMV  20.52 млн   отклонение от тренда   +2.8%
  18.08–16.09.25       GMV  22.24 млн   отклонение от тренда   +6.7%  ← всплеск
  17.09–16.10.25       GMV  22.80 млн   отклонение от тренда   +5.2%
  17.10–15.11.25       GMV  22.95 млн   отклонение от тренда   +1.7%
  16.11–15.12.25       GMV  25.52 млн   отклонение от тренда   +8.3%  ← всплеск
  16.12–14.01.26       GMV  25.36 млн   отклонение от тренда   +3.6%
  15.01–13.02.26       GMV  21.01 млн   отклонение от тренда  -19.3%  ← провал


Картина резкая и очень неудобная:

* **16.11–15.12** (11.11, «чёрная пятница», предновогодний спрос) — **+8 % к тренду**;
* **15.01–13.02.2026** — **−19 % к тренду**, самый глубокий провал за всю историю.

А это **наше валидационное окно**. То есть мы калибруем модель на худшем по спросу
месяце в данных.

## 8.3 Куда попадает боевое окно

Боевой таргет — 14.02 … 15.03.2026. Данные заканчиваются 13.02.2026, поэтому напрямую
уровень этого окна не измерить. Но у нас есть **тот же календарный интервал годом ранее**:
история начинается 01.01.2025, значит 14.02 … 15.03.2025 полностью наблюдаем.
Сравним его с предшествующим окном 15.01 … 13.02.2025 — той же парой окон,
что валидация и боевой прогноз, только на год раньше.

In [25]:
def win_sum(a, b):
    return daily_g[a:b + 1].sum()

jan_feb_25 = win_sum(14, 43)      # 15.01–13.02.2025 — аналог валидационного окна
feb_mar_25 = win_sum(44, 73)      # 14.02–15.03.2025 — аналог боевого окна
jan_feb_26 = win_sum(379, 408)    # 15.01–13.02.2026 — само валидационное окно
trend_step = np.exp(coef[0])

print(f"15.01–13.02.2025 : {jan_feb_25/1e6:6.2f} млн   (аналог валидационного окна)")
print(f"14.02–15.03.2025 : {feb_mar_25/1e6:6.2f} млн   (аналог боевого окна)")
print(f"отношение        : {feb_mar_25/jan_feb_25:.3f}")
print(f"из них тренд     : {trend_step:.3f}")
print(f"чистая сезонность: {(feb_mar_25/jan_feb_25)/trend_step:.3f}"
      f"  → боевое окно примерно на {100*((feb_mar_25/jan_feb_25)/trend_step-1):.0f}% сильнее валидационного\n")

# то же в метрической шкале: средний log1p таргета
for a, nm in [(13, "15.01–13.02.2025"), (43, "14.02–15.03.2025")]:
    ya = make_target(a)["y"].to_numpy()
    print(f"{nm}: mean log1p = {np.log1p(ya).mean():.4f}, доля нулей = {100*(ya==0).mean():.1f}%")

seg = daily_g[44:74] / np.median(daily_g[14:44])
fig = go.Figure()
line(fig, [DAY0 + timedelta(days=i) for i in range(44, 74)], seg,
     "GMV к медиане предыдущего окна", slot=0)
fig.add_hline(y=1.0, line=dict(color=AXIS, width=1.5, dash="dash"))
fig.update_yaxes(title="во сколько раз выше января")
fig.update_layout(showlegend=False, hovermode="x")
show(fig, "Календарный аналог боевого окна: 14.02–15.03.2025",
     sub="весь месяц выше января, пик — перед 8 марта",
     w=960, h=540)

15.01–13.02.2025 :  14.39 млн   (аналог валидационного окна)
14.02–15.03.2025 :  16.73 млн   (аналог боевого окна)
отношение        : 1.163
из них тренд     : 1.042
чистая сезонность: 1.116  → боевое окно примерно на 12% сильнее валидационного

15.01–13.02.2025: mean log1p = 1.5396, доля нулей = 63.3%


14.02–15.03.2025: mean log1p = 1.7154, доля нулей = 59.7%


Ответ однозначный по направлению: **боевое окно сезонно сильнее валидационного**.
Годом ранее тот же интервал дал на 16 % больше GMV, из которых около 4 % —
общий тренд, а порядка 12 % — чистая сезонность. В метрической шкале средний `log1p`
таргета вырос с 1.54 до 1.72, доля нулей упала с 63 % до 60 %.

Причина видна на дневном графике: это не один всплеск, а **равномерно приподнятый месяц**
с пиком в первых числах марта — предпраздничные покупки. Ровно тот тип сдвига,
который бьёт по калибровке сильнее всего, потому что затрагивает всех пользователей сразу.

**Что это значит — и сколько это стоит на самом деле.** Модель, откалиброванная
на валидационном окне и применённая к боевому без поправки, будет систематически
занижать. Но прежде чем паниковать, переведём смещение в метрику.

Если прогноз смещён на постоянную $\delta$ в логарифмической шкале, а в остальном
не смещён, то

$$\mathrm{RMSLE}^2_{\text{смещ}} = \mathrm{RMSLE}^2_{\text{база}} + \delta^2$$

Наша база — около 1.8. Возведение в квадрат работает против нас в неожиданную сторону:
$\delta^2$ добавляется к 3.3, а не к 1.8, поэтому корень почти не двигается. Измерим.

In [26]:
pred_ref2 = np.expm1(0.70 * np.log1p(g90 / 3))
base_r = rmsle(y, pred_ref2)
print(f"база RMSLE = {base_r:.4f};  порог значимости на public = ±{2*boot.std():.4f}\n")
print(f"{'смещение δ':>12}{'RMSLE':>10}{'цена':>10}")
for d_ in [0.05, 0.10, 0.12, 0.176, 0.25, 0.40, 0.60]:
    r_ = rmsle(y, np.expm1(np.log1p(pred_ref2) - d_))
    print(f"{d_:>12.3f}{r_:>10.4f}{r_-base_r:>+10.4f}")

# изолированно: перенос уровня между теми же окнами годом ранее
y1, y2 = make_target(13)["y"].to_numpy(), make_target(43)["y"].to_numpy()
c1, c2 = np.expm1(np.log1p(y1).mean()), np.expm1(np.log1p(y2).mean())
print(f"\nконстанта, откалиброванная на 14.02–15.03.2025 (оракул): {rmsle(y2, np.full_like(y2, c2)):.4f}")
print(f"константа, откалиброванная на предыдущем окне          : {rmsle(y2, np.full_like(y2, c1)):.4f}")
print(f"цена переноса уровня                                   : "
      f"{rmsle(y2, np.full_like(y2, c1)) - rmsle(y2, np.full_like(y2, c2)):+.4f}")

база RMSLE = 1.8361;  порог значимости на public = ±0.0111

  смещение δ     RMSLE      цена
       0.050    1.8376   +0.0014
       0.100    1.8400   +0.0038
       0.120    1.8412   +0.0051
       0.176    1.8456   +0.0095
       0.250    1.8533   +0.0172
       0.400    1.8756   +0.0394
       0.600    1.9184   +0.0822



константа, откалиброванная на 14.02–15.03.2025 (оракул): 2.2432
константа, откалиброванная на предыдущем окне          : 2.2501
цена переноса уровня                                   : +0.0069


Честный итог: **сезонная поправка стоит примерно +0.005 … +0.010 RMSLE**.
Наблюдаемый сдвиг $\delta = 0.176$ (в нём есть и тренд, и эффект неполной популяции
начала 2025) даёт +0.0095; чистая сезонная часть около 0.12 — примерно +0.005.
Прямая проверка на паре окон 2025 года даёт +0.007.

Это **порядка порога значимости лидерборда**, а не «в разы больше», как можно было бы
решить, глядя на 12 % разницы в GMV. Причина в формуле выше: при базовой ошибке 1.8
любое смещение входит в метрику квадратично и потому сильно обесценивается —
чтобы потерять хотя бы 0.04, нужно ошибиться на $\delta \approx 0.4$.

Практический вывод от этого не меняется, но меняется приоритет: поправку **стоит сделать,
потому что она почти бесплатна** (одно число), но гоняться за ней в ущерб качеству
самой модели не нужно. Основные деньги лежат в снижении ошибки на отдельном
пользователе, а не в глобальном уровне.

Оговорка к самой оценке: она получена по одному году и по популяции, которая в начале
2025 была представлена неполно (правило отбора завязано на активность зимой 2025-26).
Число «+12 %» — указание направления и порядка, а не готовый множитель.

## 8.4 Рецепт

**1. Нормализовать таргет на уровень окна.** Вместо сырого `log1p(y)` учить на

$$z_i(T) \;=\; \log(1+y_i(T)) - \ell(T), \qquad \ell(T) = \operatorname{mean}_i \log(1+y_i(T))$$

где среднее берётся по популяции `selected_users(T)`. Так все якоря приводятся
к общему уровню, и модель учится **относительному** положению пользователя,
а не абсолютному спросу конкретного месяца. При предсказании уровень возвращается:
$\hat y = \exp(\hat z + \hat\ell(T_{\text{боевой}})) - 1$.

**2. Оценить $\hat\ell$ для боевого окна отдельно.** Это единственное число,
которое нельзя выучить из признаков, и именно оно несёт сезонный риск. Три источника,
которые стоит сложить:
* уровень последнего наблюдаемого окна $\ell(\text{2026-01-14})$;
* сезонная поправка из 8.3 (аналогичный переход годом ранее);
* тренд площадки (+4 % за 30 дней).

**3. Дать модели календарь якоря.** Даже при нормализации полезно добавить признаки
самого якоря: месяц, число промо-дней в окне признаков, отклонение уровня окна признаков
от тренда. Это позволяет модели по-разному трактовать `gmv_30d`, набранный в ноябре
и в январе.

**4. Аккуратно с якорями, задевающими Новый год.** Три якоря особенные:
* **T = 2025-11-15** — таргет-окно содержит распродажи, уровень +8 % к тренду;
* **T = 2025-12-15** — таргет-окно содержит и новогодний пик, и провал после него;
* **T = 2026-01-14** — окно признаков целиком состоит из праздничного периода,
  а таргет попадает в провал.

Выбрасывать их не нужно — после нормализации по пункту 1 они становятся сопоставимыми,
а свежие якоря ценнее всего. Но проверить чувствительность стоит: обучить с ними
и без них, сравнить на валидации.

**5. Не забывать про эффект переноса спроса.** Раздел 7.1 показал, что после новогоднего
пика две недели дают лишь 72 % от двух недель до него. Значит признак `gmv_30d`,
посчитанный на декабрьском окне, завышает уровень пользователя. Длинные окна
(90 и 365 дней) от этого страдают меньше — что подтверждается и разделом 6.2,
где годовое окно дало лучший результат, чем месячное.

**6. Обе части hurdle нарезаются одинаково.** Классификатор учится на всей популяции
якоря, регрессия — **только на подмножестве с $y>0$** того же якоря. Нормализация
уровня применяется к регрессии; для классификатора аналог — учитывать, что базовая
доля покупателей у окон разная (в разделе 8.2 разброс `zero_rate` между якорями
достигает 6 процентных пунктов даже на отфильтрованной популяции).

---
# 9. Итоговая конструкция решения

### Что делать

1. **Валидация по времени, на сопоставимой популяции.** Якорь 2026-01-14, популяция —
   `selected_users(378)`, а не все 250 000. Обучающие якоря с шагом 30 дней,
   на каждом применяется тот же фильтр (раздел 3).
2. **Многооконная сборка** в духе победителя GA Customer Revenue: окно признаков
   168 дней плюс 30-дневный таргет, несколько якорей склеиваются в один обучающий набор.
   Полная схема нарезки — в разделе 8.1.
3. **Нормализовать таргет на уровень окна** (раздел 8.4). Валидационное окно —
   сезонный провал −19 % к тренду, боевое окно годом ранее было на ~12 % сильнее
   соседнего январского. Поправка даёт +0.005…+0.010 RMSLE: не решающая величина,
   но одна из немногих, которая достаётся практически даром.
4. **Двухчастная модель.** Классификатор $p(x)$ и регрессия $m(x)$, обучаемая
   **только на покупавших**. Склейка строго по формуле $\hat y = e^{p\,m} - 1$ (раздел 5.2) —
   это единственная формула, оптимальная под RMSLE.
5. **Ёмкость — в классификатор.** Карта из раздела 6.1: $p(x)$ меняется в 8 раз,
   $m(x)$ — менее чем вдвое.
6. **Признаки:** RFM-каркас (recency по активности, корзине и покупке; частоты и суммы
   за 7/14/30/60/90/180/365 дней), обязательно `дней с покупкой` и `средний чек`
   (раздел 7.4). Корзины и запросы подавать **лагами и трендами**, а не ещё одной
   90-дневной суммой: как сумма они дублируют базу, как динамика — несут своё (7.3 против 7.4).
7. **Проверять калибровку, а не AUC.** При передискретизации возвращать prior
   формулой из раздела 5.2. Финальный сабмит сверять с `sample_submit.csv`
   по доле ненулевых и среднему `log1p`.

### Чего не делать

* Не обучать регрессию в исходной шкале рублей — под RMSLE это гарантированное
  завышение по неравенству Йенсена (раздел 1.3).
* Не склеивать hurdle как $p\cdot(e^{m}-1)$ — измеримо хуже на 0.37 RMSLE (раздел 5.2).
* Не гнаться за улучшением меньше двух сигм шума public-лидерборда (раздел 1.4).
* Не дозаполнять сетку нулями: 102 млн строк не нужны, всё считается по прореженной панели.
* Не переносить важность признаков из GA Customer Revenue напрямую — там был
  cooling period в 46 дней, здесь его нет, и recency работает существенно сильнее.
* Не склеивать якоря без нормировки на уровень окна: ноябрьские распродажи
  и январский провал различаются на 27 процентных пунктов (раздел 8.2).
* Не интерпретировать `средний чек` и `долю Поиска` как «чем больше, тем лучше»:
  у обоих знак меняется при контроле активности (раздел 7.4).

### Что осталось непроверенным

* **BTYD-модели** (Pareto/NBD, BG/NBD) как источник признаков «вероятность быть живым»
  и «ожидаемое число транзакций». Распределение интервалов между визитами имеет ровно
  ту форму, под которую они построены.
* **Нейросеть по токенизированному дневному поведению** — имеет смысл как второй
  источник с иной природой ошибок, для блендинга, а не как замена бустингу.
* **Фурье-признаки:** в полном анализе они проигрывают простому профилю дня недели,
  но в связке с длинными окнами могли бы дать небольшую добавку.

In [27]:
print("=" * 64)
print("СВОДКА".center(64))
print("=" * 64)
summary = {
    "строк / пользователей":         f"{df.height:,} / {N_USERS:,}",
    "заполненность сетки":           f"{100*df.height/(N_USERS*N_DAYS):.1f}%",
    "дней с покупкой":               f"{100*n_gmv/(N_USERS*N_DAYS):.2f}% сетки",
    "нулевой таргет за 30 дней":     f"{100*(y==0).mean():.1f}%",
    "максимальный recency":          f"{r.max()} дней (следствие правила отбора)",
    "коэффициент Джини по GMV":      f"{gini:.3f}",
    "оптимальная константа c*":      f"{best_const:.2f}",
    "порог значимости на public":    f"±{2*boot.std():.4f} RMSLE (2σ)",
    "RMSLE «все нули»":              f"{rmsle(y, np.zeros_like(y)):.4f}",
    "RMSLE логики sample_submit":    f"{rmsle(y, g30):.4f}",
    "RMSLE биннинг-hurdle":          f"{rmsle(y, np.expm1(p_hat*m_hat)):.4f}",
    "цена неверной склейки hurdle":  f"+{rmsle(y, p_hat*np.expm1(m_hat)) - rmsle(y, np.expm1(p_hat*m_hat)):.4f} RMSLE",
}
for k, v in summary.items():
    print(f"  {k:<31} {v}")
print("=" * 64)

                             СВОДКА                             
  строк / пользователей           30,631,006 / 250,000
  заполненность сетки             30.0%
  дней с покупкой                 4.63% сетки
  нулевой таргет за 30 дней       45.9%
  максимальный recency            29 дней (следствие правила отбора)
  коэффициент Джини по GMV        0.700
  оптимальная константа c*        8.41
  порог значимости на public      ±0.0111 RMSLE (2σ)
  RMSLE «все нули»                3.2036
  RMSLE логики sample_submit      2.1951
  RMSLE биннинг-hurdle            1.7468
  цена неверной склейки hurdle    +0.3739 RMSLE
